# 3\.1\.1 Construct Retrospective Mobility Profile Features

This notebook builds the retrospective mobility\-profile feature table that will feed mobility\-neighborhood clustering in Chapter 3\. The clustering unit is \`Taxi Zone × pre\_post\_cp\`, so each row represents one Taxi Zone in either the pre\-congestion\-pricing or post\-congestion\-pricing period\. The goal is not to detect anomalies yet and not to build supervised\-learning features\. The goal is to create a clean, interpretable clustering matrix that summarizes each zone’s overall mobility level and recurring temporal shape using the retained mobility metrics selected in 1\.4\.3\.

The notebook uses the 1\.3\.1 analysis\-ready mobility panel as its source of truth\. It carries forward Taxi Zone identifiers, borough, policy geography class, and static spatial context fields for interpretation, while excluding Traffic, Bridge/Tunnel, Weather, lagged features, deviation features, and broader supervised\-modeling features\. Those signals may matter later, but they do not belong in the primary retrospective clustering matrix\.

In [1]:
# ---------------------------------------------------------------------
# 3.1.1 Setup: imports and display settings
# ---------------------------------------------------------------------

from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 180)

print("3.1.1 setup imports loaded.")

3.1.1 setup imports loaded.


In [2]:
# ---------------------------------------------------------------------
# 3.1.1 Setup: constants, paths, and feature contract
# ---------------------------------------------------------------------

PIPELINE_DATA_DIR = Path("pipeline_data")
MERGED_FINAL_DIR = PIPELINE_DATA_DIR / "1.3.1.final_tables"
FINAL_311_DIR = PIPELINE_DATA_DIR / "3.1.1.final_tables"

ANALYSIS_PANEL_PATH = MERGED_FINAL_DIR / "analysis_ready_mobility_panel.parquet"

RAW_CLUSTERING_FEATURE_TABLE_PATH = FINAL_311_DIR / "mobility_profile_raw_features.parquet"
SCALED_CLUSTERING_MATRIX_PATH = FINAL_311_DIR / "mobility_profile_scaled_matrix.parquet"
CLUSTERING_METADATA_PATH = FINAL_311_DIR / "mobility_profile_metadata.parquet"
SUPPORT_SUMMARY_PATH = FINAL_311_DIR / "mobility_profile_support_summary.csv"
SCALER_SUMMARY_PATH = FINAL_311_DIR / "mobility_profile_scaler_summary.csv"
FEATURE_COLUMN_LIST_PATH = FINAL_311_DIR / "mobility_profile_feature_columns.csv"

FINAL_311_DIR.mkdir(parents=True, exist_ok=True)

STUDY_START_DATE = pd.Timestamp("2023-01-01")
STUDY_END_DATE = pd.Timestamp("2026-03-31")
CONGESTION_PRICING_START_DATE = pd.Timestamp("2025-01-05")

REBUILD_CLUSTERING_INPUTS = True
SAVE_CLUSTERING_INPUTS = True

CLUSTERING_GRAIN_COLUMNS = [
    "taxi_zone_id",
    "pre_post_cp",
]

SOURCE_PANEL_GRAIN_COLUMNS = [
    "taxi_zone_id",
    "date",
    "temporal_bucket",
]

PRIMARY_CLUSTERING_METRICS = [
    "taxi_trip_count",
    "taxi_avg_trip_speed",
    "taxi_avg_trip_duration",
    "fhvhv_trip_count",
    "fhvhv_avg_trip_speed",
    "fhvhv_avg_trip_duration",
    "subway_ridership",
    "subway_transfers",
    "bus_trip_count",
    "avg_bus_speed",
]

SENSITIVITY_ONLY_METRICS = [
    "taxi_avg_trip_distance",
    "fhvhv_avg_trip_distance",
]

PERIOD_LEVEL_FEATURE_METRICS = PRIMARY_CLUSTERING_METRICS.copy()
TEMPORAL_SHAPE_FEATURE_METRICS = PRIMARY_CLUSTERING_METRICS.copy()

TEMPORAL_BUCKET_ORDER = [
    "weekday_overnight",
    "weekday_am_peak",
    "weekday_midday",
    "weekday_pm_peak",
    "weekday_evening",
    "weekend_overnight",
    "weekend_am_peak",
    "weekend_midday",
    "weekend_pm_peak",
    "weekend_evening",
]

METRIC_LABELS = {
    "taxi_trip_count": "Taxi trips",
    "taxi_avg_trip_speed": "Taxi average speed",
    "taxi_avg_trip_duration": "Taxi average duration",
    "fhvhv_trip_count": "FHVHV trips",
    "fhvhv_avg_trip_speed": "FHVHV average speed",
    "fhvhv_avg_trip_duration": "FHVHV average duration",
    "subway_ridership": "Subway ridership",
    "subway_transfers": "Subway transfers",
    "bus_trip_count": "Bus trips",
    "avg_bus_speed": "Average bus speed",
}

CLUSTERING_METADATA_COLUMNS = [
    "taxi_zone_id",
    "zone",
    "borough",
    "canonical_location_id",
    "cbd_spatial_category",
    "distance_to_cbd_miles",
    "distance_to_gateway_miles",
    "connected_neighbor_count",
    "network_betweenness_centrality",
    "network_closeness_centrality",
    "network_clustering_coefficient",
    "pre_post_cp",
]

print(
    "3.1.1 feature contract loaded: "
    f"{len(PRIMARY_CLUSTERING_METRICS)} primary metrics, "
    f"{len(TEMPORAL_BUCKET_ORDER)} temporal buckets, "
    f"final_output_dir={FINAL_311_DIR}."
)

3.1.1 feature contract loaded: 10 primary metrics, 10 temporal buckets, final_output_dir=pipeline_data/3.1.1.final_tables.


In [3]:
# ---------------------------------------------------------------------
# 3.1.1 Setup: lightweight helpers
# ---------------------------------------------------------------------

def sql_path(path: Path) -> str:
    """Return a DuckDB-safe path string."""
    return path.as_posix().replace("'", "''")


def require_columns(available_columns, required_columns, table_name):
    """Raise a clear error when an expected upstream column is unavailable."""
    missing_columns = sorted(set(required_columns) - set(available_columns))
    if missing_columns:
        raise AssertionError(
            f"{table_name} is missing required columns: {missing_columns}"
        )


def make_level_feature_name(metric: str) -> str:
    """Name a period-level clustering feature from its source metric."""
    return f"{metric}_period_mean"


def make_shape_feature_name(metric: str, temporal_bucket: str) -> str:
    """Name a normalized temporal-shape feature from metric and bucket."""
    return f"{metric}_shape_{temporal_bucket}"

print("3.1.1 setup helpers loaded.")

3.1.1 setup helpers loaded.


## 3\.1\.1\.1 Load Inputs and Lock Feature Contract

This section loads the 1\.3\.1 analysis\-ready mobility panel and confirms the source grain, temporal bucket schema, retained metric availability, and static spatial context fields needed downstream\. The retained clustering metrics and output paths are defined in setup, so this section can focus on loading and validating the actual source data\.

### Load analysis\-ready mobility panel

Load the \`Taxi Zone × Date × Temporal Bucket\` panel from 1\.3\.1 and keep only the columns needed for clustering\-profile construction\.

In [4]:
# ---------------------------------------------------------------------
# Load 1.3.1 analysis-ready panel for clustering-profile construction
# ---------------------------------------------------------------------

if not ANALYSIS_PANEL_PATH.exists():
    raise FileNotFoundError(
        f"Expected analysis-ready mobility panel does not exist: {ANALYSIS_PANEL_PATH}"
    )

con = duckdb.connect()

# Check the parquet schema before reading rows so missing upstream fields fail early.
analysis_panel_schema_df = con.execute(f"""
    DESCRIBE SELECT *
    FROM read_parquet('{sql_path(ANALYSIS_PANEL_PATH)}')
""").df()

analysis_panel_columns = analysis_panel_schema_df["column_name"].tolist()

# Keep only the fields needed for clustering profiles and interpretation metadata.
source_panel_columns = list(dict.fromkeys(
    CLUSTERING_METADATA_COLUMNS
    + [
        "date",
        "year",
        "month",
        "day_of_week",
        "daypart",
        "daypart_order",
        "temporal_bucket",
        "temporal_bucket_order",
        "observation_sequence_id",
    ]
    + PRIMARY_CLUSTERING_METRICS
))

require_columns(
    analysis_panel_columns,
    source_panel_columns,
    "1.3.1 analysis-ready mobility panel",
)

source_panel_select_sql = ",\n        ".join(f'"{column}"' for column in source_panel_columns)

# Filter to the study period here even though 1.3.1 should already enforce it.
mobility_profile_source_df = con.execute(f"""
    SELECT
        {source_panel_select_sql}
    FROM read_parquet('{sql_path(ANALYSIS_PANEL_PATH)}')
    WHERE date BETWEEN DATE '{STUDY_START_DATE.date()}' AND DATE '{STUDY_END_DATE.date()}'
""").df()

mobility_profile_source_df["date"] = pd.to_datetime(mobility_profile_source_df["date"])

source_panel_load_summary_df = pd.DataFrame(
    {
        "metric": [
            "row_count",
            "taxi_zone_count",
            "min_date",
            "max_date",
            "distinct_temporal_buckets",
            "retained_metric_count",
        ],
        "value": [
            len(mobility_profile_source_df),
            mobility_profile_source_df["taxi_zone_id"].nunique(),
            mobility_profile_source_df["date"].min(),
            mobility_profile_source_df["date"].max(),
            mobility_profile_source_df["temporal_bucket"].nunique(),
            len(PRIMARY_CLUSTERING_METRICS),
        ],
    }
)

display(source_panel_load_summary_df)

,metric,value
0,row_count,1559590
1,taxi_zone_count,263
2,min_date,2023-01-01 00:00:00
3,max_date,2026-03-31 00:00:00
4,distinct_temporal_buckets,10
5,retained_metric_count,10


### Validate source grain and required fields

Confirm that the source panel has the expected Taxi Zone, date, policy\-period, temporal\-bucket, metric, and spatial\-context fields\.

In [5]:
# ---------------------------------------------------------------------
# Validate source grain, temporal contract, and study-period bounds
# ---------------------------------------------------------------------

source_grain_duplicate_rows = (
    len(mobility_profile_source_df)
    - len(mobility_profile_source_df[SOURCE_PANEL_GRAIN_COLUMNS].drop_duplicates())
)

observed_temporal_buckets = set(mobility_profile_source_df["temporal_bucket"].dropna())
expected_temporal_buckets = set(TEMPORAL_BUCKET_ORDER)

# These checks are intentionally structural; metric quality is reviewed in the next cell.
source_contract_validation_df = pd.DataFrame(
    {
        "check": [
            "source_path_exists",
            "positive_row_count",
            "unique_taxi_zone_date_bucket_key",
            "study_period_date_bounds",
            "expected_temporal_bucket_labels",
            "non_null_temporal_bucket_order",
            "expected_cp_labels",
            "non_null_taxi_zone_id",
        ],
        "status": [
            ANALYSIS_PANEL_PATH.exists(),
            len(mobility_profile_source_df) > 0,
            source_grain_duplicate_rows == 0,
            (
                mobility_profile_source_df["date"].min() >= STUDY_START_DATE
                and mobility_profile_source_df["date"].max() <= STUDY_END_DATE
            ),
            observed_temporal_buckets == expected_temporal_buckets,
            mobility_profile_source_df["temporal_bucket_order"].notna().all(),
            set(mobility_profile_source_df["pre_post_cp"].dropna()) == {"pre_cp", "post_cp"},
            mobility_profile_source_df["taxi_zone_id"].notna().all(),
        ],
    }
)

display(source_contract_validation_df)

if not source_contract_validation_df["status"].all():
    failed_checks = source_contract_validation_df.loc[
        ~source_contract_validation_df["status"],
        "check",
    ].tolist()
    raise AssertionError(f"3.1.1 source contract validation failed: {failed_checks}")

,check,status
0,source_path_exists,True
1,positive_row_count,True
2,unique_taxi_zone_date_bucket_key,True
3,study_period_date_bounds,True
4,expected_temporal_bucket_labels,True
5,non_null_temporal_bucket_order,True
6,expected_cp_labels,True
7,non_null_taxi_zone_id,True


### Review retained clustering metric availability

Before we build profile features, quickly confirm that the 10 retained clustering metrics are present and populated in the source panel\. This is not a fresh missingness audit; it is just the handoff check that the metrics selected in 1\.4\.3 are available for 3\.1\.1\.

In [6]:
# ---------------------------------------------------------------------
# Review retained clustering metric availability
# ---------------------------------------------------------------------

metric_availability_records = []

for metric in PRIMARY_CLUSTERING_METRICS:
    metric_series = mobility_profile_source_df[metric]

    # Non-null rate confirms the metric is usable; zero rate flags structurally quiet zones/buckets.
    metric_availability_records.append(
        {
            "metric": metric,
            "metric_label": METRIC_LABELS[metric],
            "non_null_count": int(metric_series.notna().sum()),
            "non_null_pct": round(metric_series.notna().mean() * 100, 3),
            "zero_count": int((metric_series.fillna(0) == 0).sum()),
            "distinct_taxi_zones": int(
                mobility_profile_source_df.loc[
                    metric_series.notna(),
                    "taxi_zone_id",
                ].nunique()
            ),
        }
    )

retained_metric_availability_df = pd.DataFrame(metric_availability_records)

display(retained_metric_availability_df)

,metric,metric_label,non_null_count,non_null_pct,zero_count,distinct_taxi_zones
0,taxi_trip_count,Taxi trips,1531505,98.199,434452,263
1,taxi_avg_trip_speed,Taxi average speed,1125138,72.143,456085,263
2,taxi_avg_trip_duration,Taxi average duration,1125138,72.143,434452,263
3,fhvhv_trip_count,FHVHV trips,1531505,98.199,36116,263
4,fhvhv_avg_trip_speed,FHVHV average speed,1523474,97.684,36125,263
5,fhvhv_avg_trip_duration,FHVHV average duration,1523474,97.684,36116,263
6,subway_ridership,Subway ridership,912213,58.491,647377,156
7,subway_transfers,Subway transfers,912213,58.491,686311,156
8,bus_trip_count,Bus trips,1476478,94.671,83112,252
9,avg_bus_speed,Average bus speed,1476478,94.671,83112,252


Findings\. The retained clustering metrics are available enough to move forward, but they carry the expected modality\-specific footprint\. Taxi and FHVHV trip counts cover nearly the full panel, while Taxi speed and duration are present on about 72% of rows because many zone\-buckets have no Taxi activity to average\. FHVHV speed and duration are much denser at roughly 98%\. Subway metrics are only present for the 156 Taxi Zones with subway activity, which is expected rather than a data problem\. Bus metrics cover 252 zones, giving us broad but not universal surface\-transit coverage\. Overall, the 10\-metric set is usable for retrospective clustering as long as the later support policy keeps structurally absent or low\-volume cases from dominating the profile features\.

## 3.1.1.2 Build Period-Level Mobility Features

This section builds `period_level_feature_df`, the first part of the clustering representation. The table is keyed by `taxi_zone_id × pre_post_cp` and contains one period-level mean feature for each retained clustering metric. Based on 1.4.3, the mean is the right summary here because it captures each zone-period’s overall mobility level, while the temporal-shape features in the next section capture how that activity is distributed across the 10 project temporal buckets.

### Aggregate retained metrics to Taxi Zone x policy period

Collapse the source panel from \`Taxi Zone × Date × Temporal Bucket\` to the clustering row grain: one row per Taxi Zone in the pre\-period and one row per Taxi Zone in the post\-period\. Each retained metric becomes a period\-level mean feature\.

In [7]:
# ---------------------------------------------------------------------
# Build Taxi Zone x policy-period level features
# ---------------------------------------------------------------------

level_feature_name_map = {
    metric: make_level_feature_name(metric)
    for metric in PERIOD_LEVEL_FEATURE_METRICS
}

# Period means summarize overall exposure; temporal-shape cells will handle within-period timing.
period_level_feature_df = (
    mobility_profile_source_df
    .groupby(CLUSTERING_GRAIN_COLUMNS, dropna=False)[PERIOD_LEVEL_FEATURE_METRICS]
    .mean()
    .reset_index()
    .rename(columns=level_feature_name_map)
    .sort_values(CLUSTERING_GRAIN_COLUMNS)
    .reset_index(drop=True)
)

period_level_feature_columns = [
    level_feature_name_map[metric]
    for metric in PERIOD_LEVEL_FEATURE_METRICS
]

period_level_feature_summary_df = pd.DataFrame(
    {
        "metric": [
            "period_level_row_count",
            "taxi_zone_count",
            "policy_period_count",
            "level_feature_count",
            "rows_with_any_missing_level_feature",
        ],
        "value": [
            len(period_level_feature_df),
            period_level_feature_df["taxi_zone_id"].nunique(),
            period_level_feature_df["pre_post_cp"].nunique(),
            len(period_level_feature_columns),
            int(period_level_feature_df[period_level_feature_columns].isna().any(axis=1).sum()),
        ],
    }
)

display(period_level_feature_summary_df)

,metric,value
0,period_level_row_count,526
1,taxi_zone_count,263
2,policy_period_count,2
3,level_feature_count,10
4,rows_with_any_missing_level_feature,219


Findings\. The period\-level feature table has the expected shape: 263 Taxi Zones across two policy periods, producing 526 Taxi Zone × pre\_post\_cp rows and 10 level features\. The 219 rows with at least one missing level feature are not surprising, because Subway is structurally absent in many Taxi Zones and Bus is not universal either\. This confirms the table is correctly shaped, while also reinforcing that null handling and support flags need to be handled deliberately before scaling\.

### Review period\-level feature ranges

Now check the period\-level feature ranges before we move on\. This is the natural pause point: if the clustering rows, missingness, or metric ranges look odd here, we should fix that before building temporal\-shape features\.

In [8]:
# ---------------------------------------------------------------------
# Review period-level feature ranges before building shape features
# ---------------------------------------------------------------------

period_level_range_records = []

for metric in PERIOD_LEVEL_FEATURE_METRICS:
    feature_column = level_feature_name_map[metric]
    feature_series = period_level_feature_df[feature_column]

    # Ranges are rounded for readability; the raw feature table keeps full precision.
    period_level_range_records.append(
        {
            "metric": metric,
            "metric_label": METRIC_LABELS[metric],
            "non_null_zone_periods": int(feature_series.notna().sum()),
            "null_zone_periods": int(feature_series.isna().sum()),
            "min_value": round(feature_series.min(skipna=True), 3),
            "median_value": round(feature_series.median(skipna=True), 3),
            "max_value": round(feature_series.max(skipna=True), 3),
        }
    )

period_level_feature_ranges_df = pd.DataFrame(period_level_range_records)

display(period_level_feature_ranges_df)

,metric,metric_label,non_null_zone_periods,null_zone_periods,min_value,median_value,max_value
0,taxi_trip_count,Taxi trips,526,0,0.000,5.034,1121.265
1,taxi_avg_trip_speed,Taxi average speed,525,1,9.078,13.680,39.745
2,taxi_avg_trip_duration,Taxi average duration,525,1,706.594,1678.503,3759.932
3,fhvhv_trip_count,FHVHV trips,526,0,0.000,389.062,2658.749
4,fhvhv_avg_trip_speed,FHVHV average speed,525,1,11.277,16.448,27.513
5,fhvhv_avg_trip_duration,FHVHV average duration,525,1,502.524,1056.800,2670.913
6,subway_ridership,Subway ridership,309,217,11.719,2961.395,39429.038
7,subway_transfers,Subway transfers,309,217,0.039,95.362,2867.721
8,bus_trip_count,Bus trips,504,22,38.929,641.953,8889.678
9,avg_bus_speed,Average bus speed,504,22,5.210,8.675,29.077


Findings\. The period\-level features land at the expected Taxi Zone × pre/post grain, with 526 possible zone\-period rows\. Taxi and FHVHV trip counts are complete, while the average speed and duration fields each have one null zone\-period, likely where the supporting trip activity is absent or too thin to compute a valid average\. Subway is the main structural gap: only 309 zone\-periods have subway ridership and transfer values, reflecting the fact that many Taxi Zones do not have subway activity\. Bus covers most of the panel, with 504 non\-null zone\-periods\. The ranges look broadly plausible, but this confirms why the support and fill policy in 3\.1\.1\.4 is important before we scale anything for clustering\.

## 3.1.1.3 Build Temporal-Shape Mobility Features

This section builds `temporal_shape_feature_df`, the second major feature table in the clustering pipeline. The table is keyed by `taxi_zone_id` × `pre_post_cp` and contains normalized temporal-shape features for each retained metric across the 10 project temporal_bucket values. These features are meant to capture when activity or performance concentrates within a zone-period, while period_level_feature_df captures how much overall activity or performance that zone-period has.

### Aggregate retained metrics to Taxi Zone x policy period x temporal bucket

First build the bucket\-level version of the same retained metrics\. This keeps the existing project temporal buckets intact and gives us the numerator for each normalized shape feature\.

In [9]:
# ---------------------------------------------------------------------
# Build Taxi Zone x policy-period x temporal-bucket metric means
# ---------------------------------------------------------------------

bucket_group_columns = CLUSTERING_GRAIN_COLUMNS + [
    "temporal_bucket",
    "temporal_bucket_order",
]

# Bucket means preserve the current temporal schema while compressing daily rows.
temporal_bucket_metric_df = (
    mobility_profile_source_df
    .groupby(bucket_group_columns, dropna=False)[TEMPORAL_SHAPE_FEATURE_METRICS]
    .mean()
    .reset_index()
    .sort_values(CLUSTERING_GRAIN_COLUMNS + ["temporal_bucket_order"])
    .reset_index(drop=True)
)

temporal_bucket_summary_df = pd.DataFrame(
    {
        "metric": [
            "bucket_metric_row_count",
            "taxi_zone_count",
            "policy_period_count",
            "temporal_bucket_count",
            "shape_source_metric_count",
        ],
        "value": [
            len(temporal_bucket_metric_df),
            temporal_bucket_metric_df["taxi_zone_id"].nunique(),
            temporal_bucket_metric_df["pre_post_cp"].nunique(),
            temporal_bucket_metric_df["temporal_bucket"].nunique(),
            len(TEMPORAL_SHAPE_FEATURE_METRICS),
        ],
    }
)

display(temporal_bucket_summary_df)

,metric,value
0,bucket_metric_row_count,5260
1,taxi_zone_count,263
2,policy_period_count,2
3,temporal_bucket_count,10
4,shape_source_metric_count,10


Findings\. The bucket\-level source table has the expected shape: 263 Taxi Zones across two policy periods and 10 temporal buckets, producing 5,260 Taxi Zone × pre\_post\_cp × temporal\_bucket rows\. That confirms the temporal\-shape build is working off the full project bucket schema rather than collapsing or dropping buckets before normalization\.

### Convert bucket values to normalized temporal\-shape indices

Normalize each bucket against that zone\-period’s own period mean\. A value of 100 means the bucket matches the zone\-period average, values above 100 are relatively high for that zone\-period, and values below 100 are relatively low\.

In [10]:
# ---------------------------------------------------------------------
# Convert bucket means to normalized temporal-shape indices
# ---------------------------------------------------------------------

period_level_lookup_columns = CLUSTERING_GRAIN_COLUMNS + period_level_feature_columns

shape_source_df = temporal_bucket_metric_df.merge(
    period_level_feature_df[period_level_lookup_columns],
    on=CLUSTERING_GRAIN_COLUMNS,
    how="left",
)

shape_long_records = []

for metric in TEMPORAL_SHAPE_FEATURE_METRICS:
    level_column = level_feature_name_map[metric]

    metric_shape_df = shape_source_df[
        CLUSTERING_GRAIN_COLUMNS
        + ["temporal_bucket", "temporal_bucket_order", metric, level_column]
    ].copy()

    metric_shape_df["metric"] = metric
    metric_shape_df["metric_label"] = METRIC_LABELS[metric]
    metric_shape_df["period_level_value"] = metric_shape_df[level_column]
    metric_shape_df["bucket_value"] = metric_shape_df[metric]

    # Undefined denominators are kept as null here, then handled explicitly in 3.1.1.4.
    valid_denominator_mask = (
        metric_shape_df["period_level_value"].notna()
        & (metric_shape_df["period_level_value"] != 0)
    )
    metric_shape_df["shape_index"] = np.where(
        valid_denominator_mask,
        (metric_shape_df["bucket_value"] / metric_shape_df["period_level_value"]) * 100,
        np.nan,
    )

    shape_long_records.append(
        metric_shape_df[
            CLUSTERING_GRAIN_COLUMNS
            + [
                "temporal_bucket",
                "temporal_bucket_order",
                "metric",
                "metric_label",
                "bucket_value",
                "period_level_value",
                "shape_index",
            ]
        ]
    )

temporal_shape_long_df = pd.concat(shape_long_records, ignore_index=True)

temporal_shape_feature_df = period_level_feature_df[CLUSTERING_GRAIN_COLUMNS].copy()
temporal_shape_feature_columns = []

for metric in TEMPORAL_SHAPE_FEATURE_METRICS:
    metric_wide_df = (
        temporal_shape_long_df.loc[temporal_shape_long_df["metric"] == metric]
        .pivot(
            index=CLUSTERING_GRAIN_COLUMNS,
            columns="temporal_bucket",
            values="shape_index",
        )
        .reindex(columns=TEMPORAL_BUCKET_ORDER)
        .reset_index()
    )

    metric_feature_map = {
        bucket: make_shape_feature_name(metric, bucket)
        for bucket in TEMPORAL_BUCKET_ORDER
    }
    metric_wide_df = metric_wide_df.rename(columns=metric_feature_map)

    metric_shape_columns = [
        metric_feature_map[bucket]
        for bucket in TEMPORAL_BUCKET_ORDER
    ]
    temporal_shape_feature_columns.extend(metric_shape_columns)

    temporal_shape_feature_df = temporal_shape_feature_df.merge(
        metric_wide_df[CLUSTERING_GRAIN_COLUMNS + metric_shape_columns],
        on=CLUSTERING_GRAIN_COLUMNS,
        how="left",
    )

temporal_shape_summary_df = pd.DataFrame(
    {
        "metric": [
            "shape_feature_row_count",
            "taxi_zone_count",
            "policy_period_count",
            "shape_feature_count",
            "rows_with_any_missing_shape_feature",
        ],
        "value": [
            len(temporal_shape_feature_df),
            temporal_shape_feature_df["taxi_zone_id"].nunique(),
            temporal_shape_feature_df["pre_post_cp"].nunique(),
            len(temporal_shape_feature_columns),
            int(temporal_shape_feature_df[temporal_shape_feature_columns].isna().any(axis=1).sum()),
        ],
    }
)

display(temporal_shape_summary_df)

,metric,value
0,shape_feature_row_count,526
1,taxi_zone_count,263
2,policy_period_count,2
3,shape_feature_count,100
4,rows_with_any_missing_shape_feature,229


Findings\. The normalized shape table also lands at the expected clustering grain, with 526 Taxi Zone × pre\_post\_cp rows and 100 shape features: 10 retained metrics times 10 temporal buckets\. The 229 rows with at least one missing shape feature are not surprising, because undefined shape values mainly come from structurally absent Subway activity, incomplete Bus coverage, or zone\-periods where the period\-level denominator is missing or zero\. So the feature table is structurally correct, but it still needs the support and fill policy in 3\.1\.1\.4 before it becomes a clean clustering matrix\.

### Review temporal\-shape feature ranges

Review how much of each metric’s shape profile is defined and how wide the normalized bucket indices get\. This is the human\-in\-the\-loop pause before we decide which undefined shape values should be filled, flagged, or excluded\.

In [11]:
# ---------------------------------------------------------------------
# Review normalized temporal-shape ranges by metric
# ---------------------------------------------------------------------

shape_range_records = []

for metric in TEMPORAL_SHAPE_FEATURE_METRICS:
    metric_shape_df = temporal_shape_long_df.loc[
        temporal_shape_long_df["metric"] == metric
    ].copy()

    # Denominator issues usually mean structural absence or zero period-level activity.
    undefined_denominator_rows = int(
        metric_shape_df["period_level_value"].isna().sum()
        + (metric_shape_df["period_level_value"] == 0).sum()
    )

    shape_range_records.append(
        {
            "metric": metric,
            "metric_label": METRIC_LABELS[metric],
            "non_null_shape_values": int(metric_shape_df["shape_index"].notna().sum()),
            "null_shape_values": int(metric_shape_df["shape_index"].isna().sum()),
            "undefined_denominator_rows": undefined_denominator_rows,
            "min_shape_index": round(metric_shape_df["shape_index"].min(skipna=True), 3),
            "median_shape_index": round(metric_shape_df["shape_index"].median(skipna=True), 3),
            "max_shape_index": round(metric_shape_df["shape_index"].max(skipna=True), 3),
        }
    )

temporal_shape_range_summary_df = pd.DataFrame(shape_range_records)

display(temporal_shape_range_summary_df)

,metric,metric_label,non_null_shape_values,null_shape_values,undefined_denominator_rows,min_shape_index,median_shape_index,max_shape_index
0,taxi_trip_count,Taxi trips,5249,11,10,0.000,90.670,965.278
1,taxi_avg_trip_speed,Taxi average speed,5213,47,10,0.000,101.939,254.957
2,taxi_avg_trip_duration,Taxi average duration,5213,47,10,2.877,93.660,380.935
3,fhvhv_trip_count,FHVHV trips,5249,11,10,0.000,102.407,907.563
4,fhvhv_avg_trip_speed,FHVHV average speed,5237,23,10,47.478,96.995,186.116
5,fhvhv_avg_trip_duration,FHVHV average duration,5237,23,10,14.048,99.008,254.316
6,subway_ridership,Subway ridership,3090,2170,2170,3.844,68.807,396.458
7,subway_transfers,Subway transfers,3090,2170,2170,0.112,58.859,418.820
8,bus_trip_count,Bus trips,4986,274,220,1.569,81.240,304.891
9,avg_bus_speed,Average bus speed,4986,274,220,38.569,98.572,190.120


Findings\. The shape features look directionally right\. Taxi and FHVHV shapes are almost fully populated, while Subway has the largest structural gap, with 2,170 null shape values tied directly to missing or zero subway denominators in non\-subway zones\. Bus shows a smaller version of the same issue\. The median shape indices sit in a reasonable band around 100, which is what we want from a within\-zone normalization, but the maximum values get quite large for trip\-count metrics, showing that some zones concentrate a lot of their activity into a small number of buckets\. That makes the temporal\-shape features informative, but it also reinforces the need to handle structurally undefined cases explicitly before scaling\.

## 3.1.1.4 Apply Support and Inclusion Policy

This section builds `clustering_support_df`, the zone-period support table that will govern how the level and temporal-shape features are cleaned before clustering. The goal here is to make structural absence and low-support cases explicit, not to hide them. Taxi, FHVHV, Subway, and Bus each have different coverage behavior, so the notebook needs a clear support layer before the final raw clustering feature table is assembled.

### Build zone\-period support indicators

Start by making the support conditions explicit at the same \`taxi\_zone\_id × pre\_post\_cp\` grain as the clustering rows\. This support table is the place where structural Subway absence, partial Bus coverage, low\-volume surface\-mobility activity, and undefined shape denominators become visible\.

In [12]:
# ---------------------------------------------------------------------
# Build zone-period support flags for structural absence and low activity
# ---------------------------------------------------------------------

# Use conservative thresholds first; the next cells show whether they need adjustment.
LOW_TAXI_TRIP_THRESHOLD = 10
LOW_FHVHV_TRIP_THRESHOLD = 100
LOW_BUS_TRIP_THRESHOLD = 100
LOW_SUBWAY_RIDERSHIP_THRESHOLD = 100

shape_null_counts_df = (
    temporal_shape_long_df.groupby(["taxi_zone_id", "pre_post_cp", "metric"], dropna=False)
    .agg(
        null_shape_bucket_count=("shape_index", lambda s: int(s.isna().sum())),
        undefined_denominator_bucket_count=(
            "period_level_value",
            lambda s: int(s.isna().sum() + (s == 0).sum()),
        ),
    )
    .reset_index()
)

shape_null_summary_wide_df = (
    shape_null_counts_df
    .pivot(index=CLUSTERING_GRAIN_COLUMNS, columns="metric", values="null_shape_bucket_count")
    .reset_index()
)
shape_null_summary_wide_df.columns = [
    *CLUSTERING_GRAIN_COLUMNS,
    *[
        f"{metric}_shape_null_bucket_count"
        for metric in TEMPORAL_SHAPE_FEATURE_METRICS
    ],
]

shape_denominator_summary_wide_df = (
    shape_null_counts_df
    .pivot(
        index=CLUSTERING_GRAIN_COLUMNS,
        columns="metric",
        values="undefined_denominator_bucket_count",
    )
    .reset_index()
)
shape_denominator_summary_wide_df.columns = [
    *CLUSTERING_GRAIN_COLUMNS,
    *[
        f"{metric}_undefined_shape_denominator_bucket_count"
        for metric in TEMPORAL_SHAPE_FEATURE_METRICS
    ],
]

clustering_support_df = (
    period_level_feature_df
    .merge(shape_null_summary_wide_df, on=CLUSTERING_GRAIN_COLUMNS, how="left")
    .merge(shape_denominator_summary_wide_df, on=CLUSTERING_GRAIN_COLUMNS, how="left")
)

# Missing level features usually reflect true structural absence rather than random nulls.
clustering_support_df["has_taxi_level_support"] = (
    clustering_support_df[make_level_feature_name("taxi_trip_count")].fillna(0) > 0
)
clustering_support_df["has_fhvhv_level_support"] = (
    clustering_support_df[make_level_feature_name("fhvhv_trip_count")].fillna(0) > 0
)
clustering_support_df["has_subway_level_support"] = (
    clustering_support_df[make_level_feature_name("subway_ridership")].notna()
)
clustering_support_df["has_bus_level_support"] = (
    clustering_support_df[make_level_feature_name("bus_trip_count")].notna()
)

clustering_support_df["low_taxi_activity_flag"] = (
    clustering_support_df[make_level_feature_name("taxi_trip_count")].fillna(0) < LOW_TAXI_TRIP_THRESHOLD
)
clustering_support_df["low_fhvhv_activity_flag"] = (
    clustering_support_df[make_level_feature_name("fhvhv_trip_count")].fillna(0) < LOW_FHVHV_TRIP_THRESHOLD
)
clustering_support_df["low_bus_activity_flag"] = (
    clustering_support_df[make_level_feature_name("bus_trip_count")].fillna(0) < LOW_BUS_TRIP_THRESHOLD
)
clustering_support_df["low_subway_activity_flag"] = (
    clustering_support_df[make_level_feature_name("subway_ridership")].fillna(0) < LOW_SUBWAY_RIDERSHIP_THRESHOLD
)

clustering_support_df["has_any_missing_level_feature"] = clustering_support_df[
    period_level_feature_columns
].isna().any(axis=1)

# This is the broadest signal that one or more metric families need special handling.
shape_null_count_columns = [
    f"{metric}_shape_null_bucket_count"
    for metric in TEMPORAL_SHAPE_FEATURE_METRICS
]
clustering_support_df[shape_null_count_columns] = clustering_support_df[
    shape_null_count_columns
].fillna(0)
clustering_support_df["has_any_missing_shape_feature"] = (
    clustering_support_df[shape_null_count_columns].sum(axis=1) > 0
)

clustering_support_df["structural_subway_absence_flag"] = ~clustering_support_df["has_subway_level_support"]
clustering_support_df["structural_bus_absence_flag"] = ~clustering_support_df["has_bus_level_support"]
clustering_support_df["any_surface_mode_low_support_flag"] = (
    clustering_support_df[[
        "low_taxi_activity_flag",
        "low_fhvhv_activity_flag",
        "low_bus_activity_flag",
    ]].any(axis=1)
)
clustering_support_df["any_support_issue_flag"] = clustering_support_df[[
    "has_any_missing_level_feature",
    "has_any_missing_shape_feature",
    "structural_subway_absence_flag",
    "structural_bus_absence_flag",
    "any_surface_mode_low_support_flag",
]].any(axis=1)

support_summary_df = pd.DataFrame(
    {
        "metric": [
            "zone_period_rows",
            "rows_with_any_support_issue",
            "rows_with_structural_subway_absence",
            "rows_with_structural_bus_absence",
            "rows_with_low_taxi_activity",
            "rows_with_low_fhvhv_activity",
            "rows_with_low_bus_activity",
            "rows_with_any_missing_shape_feature",
        ],
        "value": [
            len(clustering_support_df),
            int(clustering_support_df["any_support_issue_flag"].sum()),
            int(clustering_support_df["structural_subway_absence_flag"].sum()),
            int(clustering_support_df["structural_bus_absence_flag"].sum()),
            int(clustering_support_df["low_taxi_activity_flag"].sum()),
            int(clustering_support_df["low_fhvhv_activity_flag"].sum()),
            int(clustering_support_df["low_bus_activity_flag"].sum()),
            int(clustering_support_df["has_any_missing_shape_feature"].sum()),
        ],
    }
)

display(support_summary_df)

,metric,value
0,zone_period_rows,526
1,rows_with_any_support_issue,376
2,rows_with_structural_subway_absence,217
3,rows_with_structural_bus_absence,22
4,rows_with_low_taxi_activity,325
5,rows_with_low_fhvhv_activity,86
6,rows_with_low_bus_activity,47
7,rows_with_any_missing_shape_feature,229


Findings\. Most zone\-period rows need some kind of support handling: 376 of 526 rows have at least one issue, leaving only 150 fully clean rows under the current first\-pass rules\. The biggest driver is structural Subway absence, which affects 217 rows, followed by low Taxi activity, which shows up in 325 rows\. Bus absence is much smaller at 22 rows, and low FHVHV support is more limited than low Taxi support\. The main takeaway is that the support problem is real, but it is not random chaos\. It is mostly being driven by a predictable mix of non\-subway zones, thin Taxi activity, and the shape\-feature nulls that follow from those structural gaps\.

### Review support overlap patterns

Summarize how the support issues overlap before we make any fill or exclusion decisions\. This is the checkpoint that tells us whether the problem is mostly structural Subway/Bus absence, mostly low\-volume surface mobility, or a smaller set of zone\-period rows with several issues at once\.

In [13]:
# ---------------------------------------------------------------------
# Summarize support-issue combinations across zone-period rows
# ---------------------------------------------------------------------

support_issue_columns = [
    "structural_subway_absence_flag",
    "structural_bus_absence_flag",
    "low_taxi_activity_flag",
    "low_fhvhv_activity_flag",
    "low_bus_activity_flag",
    "has_any_missing_shape_feature",
]

support_issue_labels = {
    "structural_subway_absence_flag": "subway_absent",
    "structural_bus_absence_flag": "bus_absent",
    "low_taxi_activity_flag": "low_taxi",
    "low_fhvhv_activity_flag": "low_fhvhv",
    "low_bus_activity_flag": "low_bus",
    "has_any_missing_shape_feature": "shape_missing",
}

# Store the pattern as a readable pipe-delimited label instead of a wide binary preview.
clustering_support_df["support_issue_pattern"] = clustering_support_df[support_issue_columns].apply(
    lambda row: " | ".join(
        support_issue_labels[column]
        for column in support_issue_columns
        if row[column]
    ) if row.any() else "no_issue",
    axis=1,
)

support_pattern_summary_df = (
    clustering_support_df.groupby("support_issue_pattern", dropna=False)
    .agg(zone_period_rows=("taxi_zone_id", "size"))
    .reset_index()
    .sort_values("zone_period_rows", ascending=False)
    .reset_index(drop=True)
)

support_pattern_summary_df["row_pct"] = (
    support_pattern_summary_df["zone_period_rows"] / len(clustering_support_df) * 100
).round(3)

display(support_pattern_summary_df.head(12))

,support_issue_pattern,zone_period_rows,row_pct
0,no_issue,150,28.517
1,low_taxi,135,25.665
2,subway_absent | low_taxi | shape_missing,102,19.392
3,subway_absent | low_taxi | low_fhvhv | shape_m...,49,9.316
4,subway_absent | shape_missing,34,6.464
5,subway_absent | bus_absent | low_taxi | low_fh...,16,3.042
6,subway_absent | low_taxi | low_fhvhv | low_bus...,10,1.901
7,low_bus | shape_missing,6,1.141
8,shape_missing,4,0.760
9,low_taxi | low_fhvhv,4,0.760


Findings\. The overlap patterns make the problem much clearer\. The single biggest non\-clean group is just low\_taxi, which means a lot of rows are otherwise usable but have thin Taxi activity\. After that, the dominant issue cluster is subway\_absent \| low\_taxi \| shape\_missing, which is exactly the kind of structural pattern we would expect in lower\-activity non\-subway zones\. In other words, most of the support burden is coming from a few recurring combinations rather than a long tail of weird edge cases\. That is good news, because it suggests we can use a fairly simple support policy instead of a complicated case\-by\-case workaround\.

In [14]:
# ---------------------------------------------------------------------
# Preview the zone-period rows with the heaviest support burden
# ---------------------------------------------------------------------

clustering_support_df["support_issue_count"] = clustering_support_df[support_issue_columns].sum(axis=1)

support_preview_columns = [
    "taxi_zone_id",
    "pre_post_cp",
    "support_issue_count",
    "support_issue_pattern",
    make_level_feature_name("taxi_trip_count"),
    make_level_feature_name("fhvhv_trip_count"),
    make_level_feature_name("subway_ridership"),
    make_level_feature_name("bus_trip_count"),
]

flagged_zone_period_preview_df = (
    clustering_support_df.loc[
        clustering_support_df["any_support_issue_flag"],
        support_preview_columns,
    ]
    .sort_values(
        ["support_issue_count", make_level_feature_name("taxi_trip_count")],
        ascending=[False, True],
    )
    .head(20)
    .reset_index(drop=True)
)

display(flagged_zone_period_preview_df)

,taxi_zone_id,pre_post_cp,support_issue_count,support_issue_pattern,taxi_trip_count_period_mean,fhvhv_trip_count_period_mean,subway_ridership_period_mean,bus_trip_count_period_mean
0,58,pre_cp,6,subway_absent | bus_absent | low_taxi | low_fh...,0.062041,41.978503,NaN,NaN
1,8,pre_cp,6,subway_absent | bus_absent | low_taxi | low_fh...,0.121481,5.479704,NaN,NaN
2,105,pre_cp,6,subway_absent | bus_absent | low_taxi | low_fh...,0.144330,0.876289,NaN,NaN
3,57,pre_cp,6,subway_absent | bus_absent | low_taxi | low_fh...,0.157823,27.815510,NaN,NaN
4,105,post_cp,6,subway_absent | bus_absent | low_taxi | low_fh...,0.160714,0.839286,NaN,NaN
5,8,post_cp,6,subway_absent | bus_absent | low_taxi | low_fh...,0.290709,5.568008,NaN,NaN
6,253,post_cp,6,subway_absent | bus_absent | low_taxi | low_fh...,0.331429,5.768312,NaN,NaN
7,253,pre_cp,6,subway_absent | bus_absent | low_taxi | low_fh...,0.345679,6.437480,NaN,NaN
8,58,post_cp,6,subway_absent | bus_absent | low_taxi | low_fh...,0.374279,47.092683,NaN,NaN
9,57,post_cp,6,subway_absent | bus_absent | low_taxi | low_fh...,0.603104,33.284701,NaN,NaN


Findings\. The worst rows are exactly the ones we would worry about least as meaningful clustering anchors: they have almost no Taxi activity, very low FHVHV activity, no Subway, no Bus, and missing shape support across multiple metric families\. These look like genuinely sparse or structurally disconnected mobility profiles rather than important zones with messy data\. That gives us a clean boundary for the next decision: the real policy question is not whether these extreme rows are weak, but whether we exclude them from the primary clustering matrix or keep them only for metadata and interpretation\.

> ✅ Decision\. We are keeping zero\-activity and structurally sparse zone\-period rows in the retrospective clustering matrix rather than filtering them out\. In this notebook, these are mostly not “bad missing values”; they are part of the mobility story\. If Taxi or FHVHV activity is zero, that means no trips happened\. If Bus or Subway is absent, that often just means the zone does not meaningfully participate in that mode\. So the job of this section is to encode those cases honestly, zero\-fill where zero is the right meaning, and carry support flags forward for interpretation\. That way, quiet or infrastructure\-light zones can still form their own legitimate cluster instead of being discarded just because they are sparse\.

### Apply clustering row retention and fill policy

Apply the actual row\-retention and fill policy here\. We are keeping all \`Taxi Zone × pre\_post\_cp\` rows in the clustering universe, zero\-filling structural absence where zero is the right meaning for the metric, and using neutral fills for performance features and undefined temporal\-shape features so sparse service does not get turned into fake low\-speed congestion\.

In [15]:
# ---------------------------------------------------------------------
# Apply clustering row-retention and fill policy
# ---------------------------------------------------------------------

COUNT_LIKE_METRICS = [
    "taxi_trip_count",
    "fhvhv_trip_count",
    "subway_ridership",
    "subway_transfers",
    "bus_trip_count",
]

PERFORMANCE_METRICS = [
    "taxi_avg_trip_speed",
    "taxi_avg_trip_duration",
    "fhvhv_avg_trip_speed",
    "fhvhv_avg_trip_duration",
    "avg_bus_speed",
]

clean_period_level_feature_df = period_level_feature_df.copy()
clean_temporal_shape_feature_df = temporal_shape_feature_df.copy()
clustering_support_df = clustering_support_df.copy()

level_fill_summary_records = []
shape_fill_summary_records = []

for metric in COUNT_LIKE_METRICS:
    feature_column = make_level_feature_name(metric)
    fill_flag_column = f"{feature_column}_zero_filled_flag"

    missing_mask = clean_period_level_feature_df[feature_column].isna()
    clean_period_level_feature_df[fill_flag_column] = missing_mask

    # For demand-like metrics, missing means structural non-participation or no observed activity.
    clean_period_level_feature_df.loc[missing_mask, feature_column] = 0.0
    clustering_support_df[fill_flag_column] = missing_mask.values

    level_fill_summary_records.append(
        {
            "feature_column": feature_column,
            "fill_strategy": "zero_fill",
            "rows_filled": int(missing_mask.sum()),
        }
    )

for metric in PERFORMANCE_METRICS:
    feature_column = make_level_feature_name(metric)
    fill_flag_column = f"{feature_column}_neutral_imputed_flag"

    missing_mask = clean_period_level_feature_df[feature_column].isna()
    neutral_fill_value = clean_period_level_feature_df[feature_column].median(skipna=True)

    clean_period_level_feature_df[fill_flag_column] = missing_mask

    # Neutral medians prevent absent service from looking like zero-speed or zero-duration distress.
    clean_period_level_feature_df.loc[missing_mask, feature_column] = neutral_fill_value
    clustering_support_df[fill_flag_column] = missing_mask.values

    level_fill_summary_records.append(
        {
            "feature_column": feature_column,
            "fill_strategy": "neutral_median_fill",
            "rows_filled": int(missing_mask.sum()),
            "fill_value": round(float(neutral_fill_value), 6),
        }
    )

for metric in TEMPORAL_SHAPE_FEATURE_METRICS:
    metric_shape_columns = [
        make_shape_feature_name(metric, bucket)
        for bucket in TEMPORAL_BUCKET_ORDER
    ]
    fill_flag_column = f"{metric}_shape_neutral_imputed_flag"

    missing_mask = clean_temporal_shape_feature_df[metric_shape_columns].isna().any(axis=1)
    clean_temporal_shape_feature_df[fill_flag_column] = missing_mask

    # A neutral shape profile is indexed to 100 in every bucket.
    clean_temporal_shape_feature_df[metric_shape_columns] = clean_temporal_shape_feature_df[
        metric_shape_columns
    ].fillna(100.0)
    clustering_support_df[fill_flag_column] = missing_mask.values

    shape_fill_summary_records.append(
        {
            "metric": metric,
            "fill_strategy": "neutral_shape_fill_100",
            "rows_with_any_bucket_filled": int(missing_mask.sum()),
        }
    )

clean_clustering_row_df = clean_period_level_feature_df[CLUSTERING_GRAIN_COLUMNS].copy()
clean_clustering_row_df["retain_for_primary_clustering"] = True
clean_clustering_row_df = clean_clustering_row_df.merge(
    clustering_support_df,
    on=CLUSTERING_GRAIN_COLUMNS,
    how="left",
)

level_fill_summary_df = pd.DataFrame(level_fill_summary_records)
shape_fill_summary_df = pd.DataFrame(shape_fill_summary_records)

policy_application_summary_df = pd.DataFrame(
    {
        "metric": [
            "retained_zone_period_rows",
            "dropped_zone_period_rows",
            "zero_filled_count_features",
            "neutral_imputed_performance_features",
            "neutral_filled_shape_metric_families",
        ],
        "value": [
            len(clean_clustering_row_df),
            int((~clean_clustering_row_df["retain_for_primary_clustering"]).sum()),
            int(level_fill_summary_df.loc[level_fill_summary_df["fill_strategy"] == "zero_fill", "rows_filled"].sum()),
            int(level_fill_summary_df.loc[level_fill_summary_df["fill_strategy"] == "neutral_median_fill", "rows_filled"].sum()),
            int(shape_fill_summary_df["rows_with_any_bucket_filled"].sum()),
        ],
    }
)

display(policy_application_summary_df)

/tmp/ipykernel_926/3958266949.py:77: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  clean_temporal_shape_feature_df[fill_flag_column] = missing_mask


,metric,value
0,retained_zone_period_rows,526
1,dropped_zone_period_rows,0
2,zero_filled_count_features,456
3,neutral_imputed_performance_features,26
4,neutral_filled_shape_metric_families,552


Findings\. The applied policy does what we wanted it to do: all 526 zone\-period rows stay in the clustering universe, and none are dropped just because they are sparse or structurally absent in one mode\. Most of the cleanup happens through zero\-filling count\-like features, which is consistent with treating no activity as part of the signal rather than as an error\. Only a small number of performance features needed neutral imputation, and the shape\-feature fills are concentrated in the structurally absent or denominator\-free cases we already identified\. In other words, the policy preserves the full city landscape while keeping sparse\-service rows from turning into misleading zero\-speed or broken\-shape profiles\.

### Validate retained clustering rows

Confirm that the cleaned clustering inputs still preserve the full \`Taxi Zone × pre\_post\_cp\` row universe, that no numeric feature columns intended for scaling still contain nulls, and that the support flags remain attached for interpretation\.

In [16]:
# ---------------------------------------------------------------------
# Validate cleaned clustering-ready rows and feature tables
# ---------------------------------------------------------------------

clean_level_feature_columns = [
    make_level_feature_name(metric)
    for metric in PRIMARY_CLUSTERING_METRICS
]

clean_shape_feature_columns = [
    make_shape_feature_name(metric, bucket)
    for metric in TEMPORAL_SHAPE_FEATURE_METRICS
    for bucket in TEMPORAL_BUCKET_ORDER
]

retained_row_duplicate_count = (
    len(clean_clustering_row_df)
    - len(clean_clustering_row_df[CLUSTERING_GRAIN_COLUMNS].drop_duplicates())
)

clean_row_validation_df = pd.DataFrame(
    {
        "check": [
            "expected_zone_period_row_count",
            "unique_zone_period_key",
            "all_rows_retained_for_primary_clustering",
            "period_level_features_have_no_nulls",
            "temporal_shape_features_have_no_nulls",
            "support_flags_attached",
            "both_policy_periods_present",
        ],
        "status": [
            len(clean_clustering_row_df) == len(period_level_feature_df),
            retained_row_duplicate_count == 0,
            clean_clustering_row_df["retain_for_primary_clustering"].all(),
            not clean_period_level_feature_df[clean_level_feature_columns].isna().any().any(),
            not clean_temporal_shape_feature_df[clean_shape_feature_columns].isna().any().any(),
            {"any_support_issue_flag", "support_issue_pattern", "support_issue_count"}.issubset(clean_clustering_row_df.columns),
            set(clean_clustering_row_df["pre_post_cp"].dropna()) == {"pre_cp", "post_cp"},
        ],
    }
)

display(clean_row_validation_df)

if not clean_row_validation_df["status"].all():
    failed_checks = clean_row_validation_df.loc[
        ~clean_row_validation_df["status"],
        "check",
    ].tolist()
    raise AssertionError(
        f"3.1.1 cleaned clustering-row validation failed: {failed_checks}"
    )

,check,status
0,expected_zone_period_row_count,True
1,unique_zone_period_key,True
2,all_rows_retained_for_primary_clustering,True
3,period_level_features_have_no_nulls,True
4,temporal_shape_features_have_no_nulls,True
5,support_flags_attached,True
6,both_policy_periods_present,True


### Preview cleaned clustering\-ready feature tables

Take a quick look at the cleaned feature tables before they get merged into the final raw clustering matrix\. This is just a visibility check so we can see the post\-policy columns and make sure the filled values and flags look reasonable\.

In [17]:
# ---------------------------------------------------------------------
# Preview cleaned period-level clustering features
# ---------------------------------------------------------------------

display(clean_period_level_feature_df.head())

,taxi_zone_id,pre_post_cp,taxi_trip_count_period_mean,taxi_avg_trip_speed_period_mean,taxi_avg_trip_duration_period_mean,fhvhv_trip_count_period_mean,fhvhv_avg_trip_speed_period_mean,fhvhv_avg_trip_duration_period_mean,subway_ridership_period_mean,subway_transfers_period_mean,bus_trip_count_period_mean,avg_bus_speed_period_mean,taxi_trip_count_period_mean_zero_filled_flag,fhvhv_trip_count_period_mean_zero_filled_flag,subway_ridership_period_mean_zero_filled_flag,subway_transfers_period_mean_zero_filled_flag,bus_trip_count_period_mean_zero_filled_flag,taxi_avg_trip_speed_period_mean_neutral_imputed_flag,taxi_avg_trip_duration_period_mean_neutral_imputed_flag,fhvhv_avg_trip_speed_period_mean_neutral_imputed_flag,fhvhv_avg_trip_duration_period_mean_neutral_imputed_flag,avg_bus_speed_period_mean_neutral_imputed_flag
0,1,post_cp,1.262963,17.680506,1346.425789,0.031481,14.749211,1551.781250,0.000000,0.000000,0.000000,8.674972,False,False,True,True,True,False,False,False,False,True
1,1,pre_cp,1.376902,13.951117,1362.594862,0.037601,11.276627,502.524390,0.000000,0.000000,0.000000,8.674972,False,False,True,True,True,False,False,False,False,True
2,2,post_cp,0.145695,25.320282,2062.576471,1.281457,19.271140,1442.146691,0.000000,0.000000,143.167295,26.097102,False,False,True,True,False,False,False,False,False,False
3,2,pre_cp,0.113692,24.796866,2191.649425,1.238386,19.445937,1509.706213,0.000000,0.000000,38.929418,29.076787,False,False,True,True,False,False,False,False,False,False
4,3,post_cp,2.700222,15.682052,2046.015604,312.309978,17.658967,899.002315,818.523167,80.810166,687.427938,10.121528,False,False,False,False,False,False,False,False,False,False


In [18]:
# ---------------------------------------------------------------------
# Preview cleaned temporal-shape clustering features
# ---------------------------------------------------------------------

display(clean_temporal_shape_feature_df.head())

,taxi_zone_id,pre_post_cp,taxi_trip_count_shape_weekday_overnight,taxi_trip_count_shape_weekday_am_peak,taxi_trip_count_shape_weekday_midday,taxi_trip_count_shape_weekday_pm_peak,taxi_trip_count_shape_weekday_evening,taxi_trip_count_shape_weekend_overnight,taxi_trip_count_shape_weekend_am_peak,taxi_trip_count_shape_weekend_midday,taxi_trip_count_shape_weekend_pm_peak,taxi_trip_count_shape_weekend_evening,taxi_avg_trip_speed_shape_weekday_overnight,taxi_avg_trip_speed_shape_weekday_am_peak,taxi_avg_trip_speed_shape_weekday_midday,taxi_avg_trip_speed_shape_weekday_pm_peak,taxi_avg_trip_speed_shape_weekday_evening,taxi_avg_trip_speed_shape_weekend_overnight,taxi_avg_trip_speed_shape_weekend_am_peak,taxi_avg_trip_speed_shape_weekend_midday,taxi_avg_trip_speed_shape_weekend_pm_peak,taxi_avg_trip_speed_shape_weekend_evening,taxi_avg_trip_duration_shape_weekday_overnight,taxi_avg_trip_duration_shape_weekday_am_peak,taxi_avg_trip_duration_shape_weekday_midday,taxi_avg_trip_duration_shape_weekday_pm_peak,taxi_avg_trip_duration_shape_weekday_evening,taxi_avg_trip_duration_shape_weekend_overnight,taxi_avg_trip_duration_shape_weekend_am_peak,taxi_avg_trip_duration_shape_weekend_midday,taxi_avg_trip_duration_shape_weekend_pm_peak,taxi_avg_trip_duration_shape_weekend_evening,fhvhv_trip_count_shape_weekday_overnight,fhvhv_trip_count_shape_weekday_am_peak,fhvhv_trip_count_shape_weekday_midday,fhvhv_trip_count_shape_weekday_pm_peak,fhvhv_trip_count_shape_weekday_evening,fhvhv_trip_count_shape_weekend_overnight,fhvhv_trip_count_shape_weekend_am_peak,fhvhv_trip_count_shape_weekend_midday,fhvhv_trip_count_shape_weekend_pm_peak,fhvhv_trip_count_shape_weekend_evening,fhvhv_avg_trip_speed_shape_weekday_overnight,fhvhv_avg_trip_speed_shape_weekday_am_peak,fhvhv_avg_trip_speed_shape_weekday_midday,fhvhv_avg_trip_speed_shape_weekday_pm_peak,fhvhv_avg_trip_speed_shape_weekday_evening,fhvhv_avg_trip_speed_shape_weekend_overnight,fhvhv_avg_trip_speed_shape_weekend_am_peak,fhvhv_avg_trip_speed_shape_weekend_midday,fhvhv_avg_trip_speed_shape_weekend_pm_peak,fhvhv_avg_trip_speed_shape_weekend_evening,fhvhv_avg_trip_duration_shape_weekday_overnight,fhvhv_avg_trip_duration_shape_weekday_am_peak,fhvhv_avg_trip_duration_shape_weekday_midday,fhvhv_avg_trip_duration_shape_weekday_pm_peak,fhvhv_avg_trip_duration_shape_weekday_evening,fhvhv_avg_trip_duration_shape_weekend_overnight,fhvhv_avg_trip_duration_shape_weekend_am_peak,fhvhv_avg_trip_duration_shape_weekend_midday,fhvhv_avg_trip_duration_shape_weekend_pm_peak,fhvhv_avg_trip_duration_shape_weekend_evening,subway_ridership_shape_weekday_overnight,subway_ridership_shape_weekday_am_peak,subway_ridership_shape_weekday_midday,subway_ridership_shape_weekday_pm_peak,subway_ridership_shape_weekday_evening,subway_ridership_shape_weekend_overnight,subway_ridership_shape_weekend_am_peak,subway_ridership_shape_weekend_midday,subway_ridership_shape_weekend_pm_peak,subway_ridership_shape_weekend_evening,subway_transfers_shape_weekday_overnight,subway_transfers_shape_weekday_am_peak,subway_transfers_shape_weekday_midday,subway_transfers_shape_weekday_pm_peak,subway_transfers_shape_weekday_evening,subway_transfers_shape_weekend_overnight,subway_transfers_shape_weekend_am_peak,subway_transfers_shape_weekend_midday,subway_transfers_shape_weekend_pm_peak,subway_transfers_shape_weekend_evening,bus_trip_count_shape_weekday_overnight,bus_trip_count_shape_weekday_am_peak,bus_trip_count_shape_weekday_midday,bus_trip_count_shape_weekday_pm_peak,bus_trip_count_shape_weekday_evening,bus_trip_count_shape_weekend_overnight,bus_trip_count_shape_weekend_am_peak,bus_trip_count_shape_weekend_midday,bus_trip_count_shape_weekend_pm_peak,bus_trip_count_shape_weekend_evening,avg_bus_speed_shape_weekday_overnight,avg_bus_speed_shape_weekday_am_peak,avg_bus_speed_shape_weekday_midday,avg_bus_speed_shape_weekday_pm_peak,avg_bus_speed_shape_weekday_evening,avg_bus_speed_shape_weekend_overnight,avg_bus_speed_shape_weekend_am_peak,avg_bus_speed_shape_w

In [19]:
# ---------------------------------------------------------------------
# Preview cleaned clustering row metadata and support flags
# ---------------------------------------------------------------------

preview_support_columns = [
    "taxi_zone_id",
    "pre_post_cp",
    "retain_for_primary_clustering",
    "any_support_issue_flag",
    "support_issue_count",
    "support_issue_pattern",
]

display(clean_clustering_row_df[preview_support_columns].head())

,taxi_zone_id,pre_post_cp,retain_for_primary_clustering,any_support_issue_flag,support_issue_count,support_issue_pattern
0,1,post_cp,True,True,6,subway_absent | bus_absent | low_taxi | low_fh...
1,1,pre_cp,True,True,6,subway_absent | bus_absent | low_taxi | low_fh...
2,2,post_cp,True,True,4,subway_absent | low_taxi | low_fhvhv | shape_m...
3,2,pre_cp,True,True,5,subway_absent | low_taxi | low_fhvhv | low_bus...
4,3,post_cp,True,True,1,low_taxi


Findings\. The cleaned clustering inputs pass the contract cleanly\. The full Taxi Zone × pre\_post\_cp row universe is still intact, the key remains unique, both policy periods are present, and the numeric feature tables no longer contain unresolved nulls\. Just as important, the support flags are still attached, so we have a clustering\-ready matrix without losing the interpretation layer that explains where structural absence or sparse\-service conditions were present\.

## 3.1.1.5 Assemble Raw Clustering Feature Table

This section combines the period-level and temporal-shape features into one raw clustering feature table at `Taxi Zone × pre_post_cp` grain. It also creates the companion metadata table used to interpret clustering results later. This follows the useful pattern from the deprecated 3.1.1 notebook: keep raw features, scaled matrix, metadata, and feature-column list separate.

### Assemble raw feature table

Join the cleaned period\-level and temporal\-shape feature families into one clustering\-ready table at \`taxi\_zone\_id × pre\_post\_cp\`\. This cell should make the resulting object visible, so the reader can see the assembled feature table itself before we move on to validation\.

In [20]:
# ---------------------------------------------------------------------
# Assemble the raw clustering feature table
# ---------------------------------------------------------------------

raw_feature_columns = CLUSTERING_GRAIN_COLUMNS + clean_level_feature_columns + clean_shape_feature_columns

# The clustering algorithm should only see the feature values and the row key.
raw_clustering_feature_df = (
    clean_period_level_feature_df[CLUSTERING_GRAIN_COLUMNS + clean_level_feature_columns]
    .merge(
        clean_temporal_shape_feature_df[CLUSTERING_GRAIN_COLUMNS + clean_shape_feature_columns],
        on=CLUSTERING_GRAIN_COLUMNS,
        how="inner",
    )
    .sort_values(CLUSTERING_GRAIN_COLUMNS)
    .reset_index(drop=True)
)

print(
    f"Raw clustering feature rows: {len(raw_clustering_feature_df):,} | "
    f"columns: {len(raw_clustering_feature_df.columns):,} | "
    f"numeric feature columns: {len(clean_level_feature_columns) + len(clean_shape_feature_columns):,}"
)

display(raw_clustering_feature_df.head())

Raw clustering feature rows: 526 | columns: 112 | numeric feature columns: 110


,taxi_zone_id,pre_post_cp,taxi_trip_count_period_mean,taxi_avg_trip_speed_period_mean,taxi_avg_trip_duration_period_mean,fhvhv_trip_count_period_mean,fhvhv_avg_trip_speed_period_mean,fhvhv_avg_trip_duration_period_mean,subway_ridership_period_mean,subway_transfers_period_mean,bus_trip_count_period_mean,avg_bus_speed_period_mean,taxi_trip_count_shape_weekday_overnight,taxi_trip_count_shape_weekday_am_peak,taxi_trip_count_shape_weekday_midday,taxi_trip_count_shape_weekday_pm_peak,taxi_trip_count_shape_weekday_evening,taxi_trip_count_shape_weekend_overnight,taxi_trip_count_shape_weekend_am_peak,taxi_trip_count_shape_weekend_midday,taxi_trip_count_shape_weekend_pm_peak,taxi_trip_count_shape_weekend_evening,taxi_avg_trip_speed_shape_weekday_overnight,taxi_avg_trip_speed_shape_weekday_am_peak,taxi_avg_trip_speed_shape_weekday_midday,taxi_avg_trip_speed_shape_weekday_pm_peak,taxi_avg_trip_speed_shape_weekday_evening,taxi_avg_trip_speed_shape_weekend_overnight,taxi_avg_trip_speed_shape_weekend_am_peak,taxi_avg_trip_speed_shape_weekend_midday,taxi_avg_trip_speed_shape_weekend_pm_peak,taxi_avg_trip_speed_shape_weekend_evening,taxi_avg_trip_duration_shape_weekday_overnight,taxi_avg_trip_duration_shape_weekday_am_peak,taxi_avg_trip_duration_shape_weekday_midday,taxi_avg_trip_duration_shape_weekday_pm_peak,taxi_avg_trip_duration_shape_weekday_evening,taxi_avg_trip_duration_shape_weekend_overnight,taxi_avg_trip_duration_shape_weekend_am_peak,taxi_avg_trip_duration_shape_weekend_midday,taxi_avg_trip_duration_shape_weekend_pm_peak,taxi_avg_trip_duration_shape_weekend_evening,fhvhv_trip_count_shape_weekday_overnight,fhvhv_trip_count_shape_weekday_am_peak,fhvhv_trip_count_shape_weekday_midday,fhvhv_trip_count_shape_weekday_pm_peak,fhvhv_trip_count_shape_weekday_evening,fhvhv_trip_count_shape_weekend_overnight,fhvhv_trip_count_shape_weekend_am_peak,fhvhv_trip_count_shape_weekend_midday,fhvhv_trip_count_shape_weekend_pm_peak,fhvhv_trip_count_shape_weekend_evening,fhvhv_avg_trip_speed_shape_weekday_overnight,fhvhv_avg_trip_speed_shape_weekday_am_peak,fhvhv_avg_trip_speed_shape_weekday_midday,fhvhv_avg_trip_speed_shape_weekday_pm_peak,fhvhv_avg_trip_speed_shape_weekday_evening,fhvhv_avg_trip_speed_shape_weekend_overnight,fhvhv_avg_trip_speed_shape_weekend_am_peak,fhvhv_avg_trip_speed_shape_weekend_midday,fhvhv_avg_trip_speed_shape_weekend_pm_peak,fhvhv_avg_trip_speed_shape_weekend_evening,fhvhv_avg_trip_duration_shape_weekday_overnight,fhvhv_avg_trip_duration_shape_weekday_am_peak,fhvhv_avg_trip_duration_shape_weekday_midday,fhvhv_avg_trip_duration_shape_weekday_pm_peak,fhvhv_avg_trip_duration_shape_weekday_evening,fhvhv_avg_trip_duration_shape_weekend_overnight,fhvhv_avg_trip_duration_shape_weekend_am_peak,fhvhv_avg_trip_duration_shape_weekend_midday,fhvhv_avg_trip_duration_shape_weekend_pm_peak,fhvhv_avg_trip_duration_shape_weekend_evening,subway_ridership_shape_weekday_overnight,subway_ridership_shape_weekday_am_peak,subway_ridership_shape_weekday_midday,subway_ridership_shape_weekday_pm_peak,subway_ridership_shape_weekday_evening,subway_ridership_shape_weekend_overnight,subway_ridership_shape_weekend_am_peak,subway_ridership_shape_weekend_midday,subway_ridership_shape_weekend_pm_peak,subway_ridership_shape_weekend_evening,subway_transfers_shape_weekday_overnight,subway_transfers_shape_weekday_am_peak,subway_transfers_shape_weekday_midday,subway_transfers_shape_weekday_pm_peak,subway_transfers_shape_weekday_evening,subway_transfers_shape_weekend_overnight,subway_transfers_shape_weekend_am_peak,subway_transfers_shape_weekend_midday,subway_transfers_shape_weekend_pm_peak,subway_transfers_shape_weekend_evening,bus_trip_count_shape_weekday_overnight,bus_trip_count_shape_weekday_am_peak,bus_trip_count_shape_weekday_midday,bus_trip_count_shape_weekday_pm_peak,bus_trip_count_shape_weekday_evening,bus_trip_count_shape_weekend_overnight,bus_trip_count_shape_weekend_am_peak,bus_trip_count_shape_weekend_midday,bus_trip_count_shape_weekend_pm_peak,bus_trip_

Findings\. The raw clustering feature table now has the shape we wanted: one row per Taxi Zone × pre\_post\_cp, with the period\-level features and temporal\-shape features fully merged into a single clustering\-ready matrix\. At this point the structure is finally simple: the level features capture how much mobility a zone\-period has overall, and the shape features capture when that activity tends to concentrate\. The table is wide, but in a good way: it is the full retrospective mobility profile that Chapter 3 clustering will actually consume\.

### Assemble clustering metadata table

Build the companion metadata table at the same \`taxi\_zone\_id × pre\_post\_cp\` grain\. This cell should also show the resulting object directly, so the reader can see which context, support, and imputation fields travel alongside the clustering rows\.

In [21]:
# ---------------------------------------------------------------------
# Assemble the clustering metadata table
# ---------------------------------------------------------------------

static_metadata_columns = [
    column for column in CLUSTERING_METADATA_COLUMNS
    if column not in CLUSTERING_GRAIN_COLUMNS
]

support_metadata_columns = [
    column for column in clean_clustering_row_df.columns
    if column not in CLUSTERING_GRAIN_COLUMNS + ["retain_for_primary_clustering"]
]

# Static zone metadata should be invariant within each zone-period row, so the first non-null value is enough.
metadata_source_df = (
    mobility_profile_source_df[CLUSTERING_GRAIN_COLUMNS + static_metadata_columns]
    .sort_values(CLUSTERING_GRAIN_COLUMNS)
    .groupby(CLUSTERING_GRAIN_COLUMNS, dropna=False)[static_metadata_columns]
    .agg(lambda s: s.dropna().iloc[0] if not s.dropna().empty else np.nan)
    .reset_index()
)

clustering_metadata_df = (
    clean_clustering_row_df[CLUSTERING_GRAIN_COLUMNS + ["retain_for_primary_clustering"] + support_metadata_columns]
    .merge(
        metadata_source_df,
        on=CLUSTERING_GRAIN_COLUMNS,
        how="left",
    )
    .sort_values(CLUSTERING_GRAIN_COLUMNS)
    .reset_index(drop=True)
)

print(
    f"Clustering metadata rows: {len(clustering_metadata_df):,} | "
    f"columns: {len(clustering_metadata_df.columns):,} | "
    f"static metadata fields: {len(static_metadata_columns):,} | "
    f"support/imputation fields: {len(support_metadata_columns) + 1:,}"
)

display(clustering_metadata_df.head())

Clustering metadata rows: 526 | columns: 79 | static metadata fields: 10 | support/imputation fields: 67


,taxi_zone_id,pre_post_cp,retain_for_primary_clustering,taxi_trip_count_period_mean,taxi_avg_trip_speed_period_mean,taxi_avg_trip_duration_period_mean,fhvhv_trip_count_period_mean,fhvhv_avg_trip_speed_period_mean,fhvhv_avg_trip_duration_period_mean,subway_ridership_period_mean,subway_transfers_period_mean,bus_trip_count_period_mean,avg_bus_speed_period_mean,taxi_trip_count_shape_null_bucket_count,taxi_avg_trip_speed_shape_null_bucket_count,taxi_avg_trip_duration_shape_null_bucket_count,fhvhv_trip_count_shape_null_bucket_count,fhvhv_avg_trip_speed_shape_null_bucket_count,fhvhv_avg_trip_duration_shape_null_bucket_count,subway_ridership_shape_null_bucket_count,subway_transfers_shape_null_bucket_count,bus_trip_count_shape_null_bucket_count,avg_bus_speed_shape_null_bucket_count,taxi_trip_count_undefined_shape_denominator_bucket_count,taxi_avg_trip_speed_undefined_shape_denominator_bucket_count,taxi_avg_trip_duration_undefined_shape_denominator_bucket_count,fhvhv_trip_count_undefined_shape_denominator_bucket_count,fhvhv_avg_trip_speed_undefined_shape_denominator_bucket_count,fhvhv_avg_trip_duration_undefined_shape_denominator_bucket_count,subway_ridership_undefined_shape_denominator_bucket_count,subway_transfers_undefined_shape_denominator_bucket_count,bus_trip_count_undefined_shape_denominator_bucket_count,avg_bus_speed_undefined_shape_denominator_bucket_count,has_taxi_level_support,has_fhvhv_level_support,has_subway_level_support,has_bus_level_support,low_taxi_activity_flag,low_fhvhv_activity_flag,low_bus_activity_flag,low_subway_activity_flag,has_any_missing_level_feature,has_any_missing_shape_feature,structural_subway_absence_flag,structural_bus_absence_flag,any_surface_mode_low_support_flag,any_support_issue_flag,support_issue_pattern,support_issue_count,taxi_trip_count_period_mean_zero_filled_flag,fhvhv_trip_count_period_mean_zero_filled_flag,subway_ridership_period_mean_zero_filled_flag,subway_transfers_period_mean_zero_filled_flag,bus_trip_count_period_mean_zero_filled_flag,taxi_avg_trip_speed_period_mean_neutral_imputed_flag,taxi_avg_trip_duration_period_mean_neutral_imputed_flag,fhvhv_avg_trip_speed_period_mean_neutral_imputed_flag,fhvhv_avg_trip_duration_period_mean_neutral_imputed_flag,avg_bus_speed_period_mean_neutral_imputed_flag,taxi_trip_count_shape_neutral_imputed_flag,taxi_avg_trip_speed_shape_neutral_imputed_flag,taxi_avg_trip_duration_shape_neutral_imputed_flag,fhvhv_trip_count_shape_neutral_imputed_flag,fhvhv_avg_trip_speed_shape_neutral_imputed_flag,fhvhv_avg_trip_duration_shape_neutral_imputed_flag,subway_ridership_shape_neutral_imputed_flag,subway_transfers_shape_neutral_imputed_flag,bus_trip_count_shape_neutral_imputed_flag,avg_bus_speed_shape_neutral_imputed_flag,zone,borough,canonical_location_id,cbd_spatial_category,distance_to_cbd_miles,distance_to_gateway_miles,connected_neighbor_count,network_betweenness_centrality,network_closeness_centrality,network_clustering_coefficient
0,1,post_cp,True,1.262963,17.680506,1346.425789,0.031481,14.749211,1551.781250,NaN,NaN,NaN,NaN,10,10,4,4,0,10,10,0,0,0,10,10,0,0,0,10,10,0,0,0,True,True,False,False,True,True,True,True,True,True,True,True,True,True,subway_absent | bus_absent | low_taxi | low_fh...,6,False,False,True,True,True,False,False,False,False,True,False,False,False,False,True,True,True,True,True,True,Newark Airport,EWR,1,gateway_to_cbd,6.633,0.000,8.0,0.275125,0.145036,0.035714
1,1,pre_cp,True,1.376902,13.951117,1362.594862,0.037601,11.276627,502.524390,NaN,NaN,NaN,NaN,10,10,2,2,0,10,10,0,0,0,10,10,0,0,0,10,10,0,0,0,True,True,False,False,True,True,True,True,True,True,True,True,True,True,subway_absent | bus_absent | low_taxi | low_fh...,6,False,False,True,True,True,False,False,False,False,True,False,False,False,False,True,True,True,True,True,True,Newark Airport,EWR,1,gateway_to_cbd,6.633,0.000,8.0,0.275125,0.145036,0.035714
2,2,post_cp,True,0.145695,25.320282,2062.576471,1.281457,19.271140,1442.146691,NaN,NaN,143.167295,26.097102,0,0,0,0,0,10,10,0,0,0,0,0,0,0,

Findings\. The metadata table cleanly separates interpretation fields from the clustering feature matrix\. It keeps the same Taxi Zone × pre\_post\_cp grain, but carries the static Taxi Zone context, support flags, and imputation flags that explain how each clustering row should be read\. That separation matters because it lets us preserve the story of structural absence and sparse\-service conditions without mixing those fields into the numeric clustering input itself\.

### Validate raw feature table

Check that the assembled raw feature table and metadata table still line up one\-to\-one, that the feature matrix is fully numeric and null\-free where it needs to be, and that we have a quick read on whether any raw feature columns are constant before scaling\.

In [22]:
# ---------------------------------------------------------------------
# Validate assembled raw clustering inputs
# ---------------------------------------------------------------------

raw_feature_duplicate_count = (
    len(raw_clustering_feature_df)
    - len(raw_clustering_feature_df[CLUSTERING_GRAIN_COLUMNS].drop_duplicates())
)
metadata_duplicate_count = (
    len(clustering_metadata_df)
    - len(clustering_metadata_df[CLUSTERING_GRAIN_COLUMNS].drop_duplicates())
)

constant_feature_count = int(
    (raw_clustering_feature_df[clean_level_feature_columns + clean_shape_feature_columns].nunique(dropna=False) <= 1).sum()
)

raw_feature_validation_df = pd.DataFrame(
    {
        "check": [
            "raw_feature_row_count_matches_clean_rows",
            "metadata_row_count_matches_raw_features",
            "raw_feature_unique_zone_period_key",
            "metadata_unique_zone_period_key",
            "raw_feature_and_metadata_keys_align",
            "raw_feature_columns_have_no_nulls",
            "metadata_retain_flag_present",
            "constant_feature_count_profiled",
        ],
        "status": [
            len(raw_clustering_feature_df) == len(clean_clustering_row_df),
            len(clustering_metadata_df) == len(raw_clustering_feature_df),
            raw_feature_duplicate_count == 0,
            metadata_duplicate_count == 0,
            raw_clustering_feature_df[CLUSTERING_GRAIN_COLUMNS].equals(
                clustering_metadata_df[CLUSTERING_GRAIN_COLUMNS]
            ),
            not raw_clustering_feature_df[clean_level_feature_columns + clean_shape_feature_columns].isna().any().any(),
            "retain_for_primary_clustering" in clustering_metadata_df.columns,
            constant_feature_count >= 0,
        ],
    }
)

display(raw_feature_validation_df)

if not raw_feature_validation_df["status"].all():
    failed_checks = raw_feature_validation_df.loc[
        ~raw_feature_validation_df["status"],
        "check",
    ].tolist()
    raise AssertionError(
        f"3.1.1 raw clustering feature validation failed: {failed_checks}"
    )

,check,status
0,raw_feature_row_count_matches_clean_rows,True
1,metadata_row_count_matches_raw_features,True
2,raw_feature_unique_zone_period_key,True
3,metadata_unique_zone_period_key,True
4,raw_feature_and_metadata_keys_align,True
5,raw_feature_columns_have_no_nulls,True
6,metadata_retain_flag_present,True
7,constant_feature_count_profiled,True


Findings\. The assembled raw inputs pass the final pre\-scaling check cleanly\. The raw feature matrix and metadata table stay perfectly aligned at the zone\-period key, the numeric clustering features are fully populated after the fill policy, and the support context is still attached on the metadata side\. So by the end of this section, we are no longer working with intermediate feature fragments; we have a complete raw clustering input package that is ready to be standardized in the next step\.

## 3.1.1.6 Scale and Validate Clustering Matrix

This section creates the scaled matrix that 3.2.1 will use to generate candidate mobility-neighborhood clusters. Scaling is applied only to numeric clustering features, not metadata. The scaler output should be explicit enough that cluster interpretation can trace back to raw feature definitions.

### Define scaling inputs

Lock the exact numeric feature columns that will be standardized for clustering\. This step should stay simple: scale the level and temporal\-shape features, but leave identifiers and metadata alone\.

In [23]:
# ---------------------------------------------------------------------
# Define the numeric feature columns that enter the clustering matrix
# ---------------------------------------------------------------------

# Only the actual clustering features get standardized.
scaling_feature_columns = clean_level_feature_columns + clean_shape_feature_columns

scaling_input_summary_df = pd.DataFrame(
    {
        "metric": [
            "row_count",
            "level_feature_count",
            "shape_feature_count",
            "total_scaling_feature_count",
        ],
        "value": [
            len(raw_clustering_feature_df),
            len(clean_level_feature_columns),
            len(clean_shape_feature_columns),
            len(scaling_feature_columns),
        ],
    }
)

display(scaling_input_summary_df)

,metric,value
0,row_count,526
1,level_feature_count,10
2,shape_feature_count,100
3,total_scaling_feature_count,110


Findings\. The clustering matrix going into scaling is the size we expected: 526 Taxi Zone × policy\-period rows, with 10 period\-level features and 100 temporal\-shape features for 110 numeric clustering inputs in total\. That confirms the feature build is structurally complete before standardization and that the final clustering representation is being driven mostly by temporal\-shape detail rather than just overall level summaries\.

### Fit scaler and transform clustering features

Fit the scaler on the raw clustering features and build \`scaled\_clustering\_matrix\_df\`, keeping the same Taxi Zone × policy\-period row order as the raw matrix\.

In [24]:
# ---------------------------------------------------------------------
# Fit the clustering scaler and build the scaled feature matrix
# ---------------------------------------------------------------------

clustering_scaler = StandardScaler()

# Fit on the raw numeric features only; metadata stays outside the matrix.
scaled_feature_array = clustering_scaler.fit_transform(
    raw_clustering_feature_df[scaling_feature_columns]
)

scaled_feature_df = pd.DataFrame(
    scaled_feature_array,
    columns=scaling_feature_columns,
    index=raw_clustering_feature_df.index,
)

scaled_clustering_matrix_df = pd.concat(
    [
        raw_clustering_feature_df[CLUSTERING_GRAIN_COLUMNS].reset_index(drop=True),
        scaled_feature_df.reset_index(drop=True),
    ],
    axis=1,
)

# Keep a compact scaler summary for downstream inspection and export.
scaler_summary_df = pd.DataFrame(
    {
        "feature_column": scaling_feature_columns,
        "raw_mean": clustering_scaler.mean_,
        "raw_scale": clustering_scaler.scale_,
    }
)

print(
    f"Scaled clustering rows: {len(scaled_clustering_matrix_df):,} | "
    f"columns: {len(scaled_clustering_matrix_df.columns):,}"
)

display(scaled_clustering_matrix_df.head())

Scaled clustering rows: 526 | columns: 112


,taxi_zone_id,pre_post_cp,taxi_trip_count_period_mean,taxi_avg_trip_speed_period_mean,taxi_avg_trip_duration_period_mean,fhvhv_trip_count_period_mean,fhvhv_avg_trip_speed_period_mean,fhvhv_avg_trip_duration_period_mean,subway_ridership_period_mean,subway_transfers_period_mean,bus_trip_count_period_mean,avg_bus_speed_period_mean,taxi_trip_count_shape_weekday_overnight,taxi_trip_count_shape_weekday_am_peak,taxi_trip_count_shape_weekday_midday,taxi_trip_count_shape_weekday_pm_peak,taxi_trip_count_shape_weekday_evening,taxi_trip_count_shape_weekend_overnight,taxi_trip_count_shape_weekend_am_peak,taxi_trip_count_shape_weekend_midday,taxi_trip_count_shape_weekend_pm_peak,taxi_trip_count_shape_weekend_evening,taxi_avg_trip_speed_shape_weekday_overnight,taxi_avg_trip_speed_shape_weekday_am_peak,taxi_avg_trip_speed_shape_weekday_midday,taxi_avg_trip_speed_shape_weekday_pm_peak,taxi_avg_trip_speed_shape_weekday_evening,taxi_avg_trip_speed_shape_weekend_overnight,taxi_avg_trip_speed_shape_weekend_am_peak,taxi_avg_trip_speed_shape_weekend_midday,taxi_avg_trip_speed_shape_weekend_pm_peak,taxi_avg_trip_speed_shape_weekend_evening,taxi_avg_trip_duration_shape_weekday_overnight,taxi_avg_trip_duration_shape_weekday_am_peak,taxi_avg_trip_duration_shape_weekday_midday,taxi_avg_trip_duration_shape_weekday_pm_peak,taxi_avg_trip_duration_shape_weekday_evening,taxi_avg_trip_duration_shape_weekend_overnight,taxi_avg_trip_duration_shape_weekend_am_peak,taxi_avg_trip_duration_shape_weekend_midday,taxi_avg_trip_duration_shape_weekend_pm_peak,taxi_avg_trip_duration_shape_weekend_evening,fhvhv_trip_count_shape_weekday_overnight,fhvhv_trip_count_shape_weekday_am_peak,fhvhv_trip_count_shape_weekday_midday,fhvhv_trip_count_shape_weekday_pm_peak,fhvhv_trip_count_shape_weekday_evening,fhvhv_trip_count_shape_weekend_overnight,fhvhv_trip_count_shape_weekend_am_peak,fhvhv_trip_count_shape_weekend_midday,fhvhv_trip_count_shape_weekend_pm_peak,fhvhv_trip_count_shape_weekend_evening,fhvhv_avg_trip_speed_shape_weekday_overnight,fhvhv_avg_trip_speed_shape_weekday_am_peak,fhvhv_avg_trip_speed_shape_weekday_midday,fhvhv_avg_trip_speed_shape_weekday_pm_peak,fhvhv_avg_trip_speed_shape_weekday_evening,fhvhv_avg_trip_speed_shape_weekend_overnight,fhvhv_avg_trip_speed_shape_weekend_am_peak,fhvhv_avg_trip_speed_shape_weekend_midday,fhvhv_avg_trip_speed_shape_weekend_pm_peak,fhvhv_avg_trip_speed_shape_weekend_evening,fhvhv_avg_trip_duration_shape_weekday_overnight,fhvhv_avg_trip_duration_shape_weekday_am_peak,fhvhv_avg_trip_duration_shape_weekday_midday,fhvhv_avg_trip_duration_shape_weekday_pm_peak,fhvhv_avg_trip_duration_shape_weekday_evening,fhvhv_avg_trip_duration_shape_weekend_overnight,fhvhv_avg_trip_duration_shape_weekend_am_peak,fhvhv_avg_trip_duration_shape_weekend_midday,fhvhv_avg_trip_duration_shape_weekend_pm_peak,fhvhv_avg_trip_duration_shape_weekend_evening,subway_ridership_shape_weekday_overnight,subway_ridership_shape_weekday_am_peak,subway_ridership_shape_weekday_midday,subway_ridership_shape_weekday_pm_peak,subway_ridership_shape_weekday_evening,subway_ridership_shape_weekend_overnight,subway_ridership_shape_weekend_am_peak,subway_ridership_shape_weekend_midday,subway_ridership_shape_weekend_pm_peak,subway_ridership_shape_weekend_evening,subway_transfers_shape_weekday_overnight,subway_transfers_shape_weekday_am_peak,subway_transfers_shape_weekday_midday,subway_transfers_shape_weekday_pm_peak,subway_transfers_shape_weekday_evening,subway_transfers_shape_weekend_overnight,subway_transfers_shape_weekend_am_peak,subway_transfers_shape_weekend_midday,subway_transfers_shape_weekend_pm_peak,subway_transfers_shape_weekend_evening,bus_trip_count_shape_weekday_overnight,bus_trip_count_shape_weekday_am_peak,bus_trip_count_shape_weekday_midday,bus_trip_count_shape_weekday_pm_peak,bus_trip_count_shape_weekday_evening,bus_trip_count_shape_weekend_overnight,bus_trip_count_shape_weekend_am_peak,bus_trip_count_shape_weekend_midday,bus_trip_count_shape_weekend_pm_peak,bus_trip_

Findings\. The scaled clustering matrix is now in its final modeling shape: the same 526 Taxi Zone × pre/post rows are preserved, but the 110 numeric clustering features have been standardized onto a common scale so later clustering is driven by relative pattern rather than raw unit size\. At this point, the matrix is no longer just a feature assembly artifact; it is the actual clustering input that 3\.2\.1 can use to generate candidate mobility\-neighborhood solutions\.

### Validate scaled matrix

Make sure the scaled matrix is still aligned to the raw rows, contains only finite numeric values, and behaves like a proper standardized input for clustering\.

In [25]:
# ---------------------------------------------------------------------
# Validate the scaled clustering matrix
# ---------------------------------------------------------------------

raw_feature_std = raw_clustering_feature_df[scaling_feature_columns].std(ddof=0)
nonconstant_feature_columns = raw_feature_std[raw_feature_std > 0].index.tolist()

scaled_feature_means = scaled_clustering_matrix_df[scaling_feature_columns].mean()
scaled_feature_std = scaled_clustering_matrix_df[scaling_feature_columns].std(ddof=0)

# Constant raw features legitimately scale to all zeros, so only nonconstant columns should have unit spread.
scaled_matrix_validation_df = pd.DataFrame(
    {
        "check": [
            "scaled_row_count_matches_raw_features",
            "scaled_key_alignment_matches_raw_features",
            "scaled_feature_count_matches_contract",
            "scaled_features_have_no_nulls",
            "scaled_features_are_finite",
            "nonconstant_features_are_centered_near_zero",
            "nonconstant_features_have_unit_std",
        ],
        "status": [
            len(scaled_clustering_matrix_df) == len(raw_clustering_feature_df),
            scaled_clustering_matrix_df[CLUSTERING_GRAIN_COLUMNS].equals(
                raw_clustering_feature_df[CLUSTERING_GRAIN_COLUMNS]
            ),
            len(scaling_feature_columns) == len(clean_level_feature_columns) + len(clean_shape_feature_columns),
            not scaled_clustering_matrix_df[scaling_feature_columns].isna().any().any(),
            np.isfinite(scaled_clustering_matrix_df[scaling_feature_columns].to_numpy()).all(),
            np.allclose(
                scaled_feature_means[nonconstant_feature_columns].to_numpy(),
                0.0,
                atol=1e-9,
            ) if nonconstant_feature_columns else True,
            np.allclose(
                scaled_feature_std[nonconstant_feature_columns].to_numpy(),
                1.0,
                atol=1e-9,
            ) if nonconstant_feature_columns else True,
        ],
    }
)

display(scaled_matrix_validation_df)

if not scaled_matrix_validation_df["status"].all():
    failed_checks = scaled_matrix_validation_df.loc[
        ~scaled_matrix_validation_df["status"],
        "check",
    ].tolist()
    raise AssertionError(
        f"3.1.1 scaled clustering-matrix validation failed: {failed_checks}"
    )

,check,status
0,scaled_row_count_matches_raw_features,True
1,scaled_key_alignment_matches_raw_features,True
2,scaled_feature_count_matches_contract,True
3,scaled_features_have_no_nulls,True
4,scaled_features_are_finite,True
5,nonconstant_features_are_centered_near_zero,True
6,nonconstant_features_have_unit_std,True


Findings\. The scaled matrix passes cleanly\. Row count and key alignment still match the raw feature table, the full 110\-feature contract is intact, and the standardized values contain no nulls or non\-finite values\. The nonconstant features are also centered near zero with unit spread, which means the matrix is in the right shape for distance\-based clustering without any one raw metric dominating just because of its original scale\.

### Summarize low\-information features

Flag the features that may carry very little separation power before clustering\. We are not dropping them here; we just want a clean read on what 3\.2\.1 may want to revisit\.

In [26]:
# ---------------------------------------------------------------------
# Profile low-information raw features before clustering
# ---------------------------------------------------------------------

raw_feature_nunique = raw_clustering_feature_df[scaling_feature_columns].nunique(dropna=False)
raw_feature_std = raw_clustering_feature_df[scaling_feature_columns].std(ddof=0)

low_information_feature_summary_df = pd.DataFrame(
    {
        "feature_column": scaling_feature_columns,
        "non_null_count": raw_clustering_feature_df[scaling_feature_columns].notna().sum().values,
        "distinct_value_count": raw_feature_nunique.values,
        "raw_std": raw_feature_std.values,
    }
)

low_information_feature_summary_df["zero_variance_flag"] = (
    low_information_feature_summary_df["raw_std"].fillna(0) == 0
)
low_information_feature_summary_df["two_or_fewer_values_flag"] = (
    low_information_feature_summary_df["distinct_value_count"] <= 2
)
low_information_feature_summary_df = low_information_feature_summary_df.sort_values(
    ["zero_variance_flag", "two_or_fewer_values_flag", "raw_std", "feature_column"],
    ascending=[False, False, True, True],
).reset_index(drop=True)

display(low_information_feature_summary_df.head(20))

,feature_column,non_null_count,distinct_value_count,raw_std,zero_variance_flag,two_or_fewer_values_flag
0,fhvhv_avg_trip_speed_period_mean,526,525,2.765568,False,False
1,avg_bus_speed_period_mean,526,505,3.067196,False,False
2,taxi_avg_trip_speed_period_mean,526,525,3.751424,False,False
3,fhvhv_avg_trip_speed_shape_weekday_evening,526,525,4.276288,False,False
4,fhvhv_avg_trip_speed_shape_weekday_midday,526,525,4.324152,False,False
5,fhvhv_avg_trip_speed_shape_weekend_midday,526,525,4.388037,False,False
6,fhvhv_avg_trip_speed_shape_weekday_pm_peak,526,525,4.700644,False,False
7,avg_bus_speed_shape_weekday_evening,526,500,5.490819,False,False
8,fhvhv_avg_trip_duration_shape_weekday_evening,526,525,6.106611,False,False
9,fhvhv_avg_trip_duration_shape_weekend_evening,526,523,6.210765,False,False


Findings\. The preview does not show any obvious low\-information problem features at the top of the matrix\. These columns all have full non\-null support, high distinct\-value counts, and meaningful raw variation, so they are carrying real separation signal rather than behaving like near\-constant filler\. In other words, at least from this first pass, the retained clustering features look live enough to justify keeping the full representation as we move into clustering\.

## 3.1.1.7 Export Clustering Inputs

This section writes the clustering inputs that the rest of Chapter 3 will actually use. The goal is simple: export the raw feature table, the scaled clustering matrix, the metadata table, and a few compact companion artifacts so downstream notebooks can run without rebuilding any feature logic from scratch.
### Write clustering feature outputs

Export the raw clustering feature table, scaled clustering matrix, and metadata table.

In [27]:
# ---------------------------------------------------------------------
# Export the core clustering input tables
# ---------------------------------------------------------------------

if SAVE_CLUSTERING_INPUTS:
    raw_clustering_feature_df.to_parquet(
        RAW_CLUSTERING_FEATURE_TABLE_PATH,
        index=False,
    )
    scaled_clustering_matrix_df.to_parquet(
        SCALED_CLUSTERING_MATRIX_PATH,
        index=False,
    )
    clustering_metadata_df.to_parquet(
        CLUSTERING_METADATA_PATH,
        index=False,
    )

print("Core clustering tables exported.")

exported_core_tables_df = pd.DataFrame(
    {
        "artifact": [
            "raw_clustering_feature_table",
            "scaled_clustering_matrix",
            "clustering_metadata",
        ],
        "path": [
            str(RAW_CLUSTERING_FEATURE_TABLE_PATH),
            str(SCALED_CLUSTERING_MATRIX_PATH),
            str(CLUSTERING_METADATA_PATH),
        ],
        "exists_after_run": [
            RAW_CLUSTERING_FEATURE_TABLE_PATH.exists(),
            SCALED_CLUSTERING_MATRIX_PATH.exists(),
            CLUSTERING_METADATA_PATH.exists(),
        ],
    }
)

display(exported_core_tables_df)

Core clustering tables exported.


,artifact,path,exists_after_run
0,raw_clustering_feature_table,pipeline_data/3.1.1.final_tables/mobility_prof...,True
1,scaled_clustering_matrix,pipeline_data/3.1.1.final_tables/mobility_prof...,True
2,clustering_metadata,pipeline_data/3.1.1.final_tables/mobility_prof...,True


### Write support and scaler summaries

Export compact support\-policy diagnostics and scaler parameters so 3\.2\.1 can inspect input quality and interpret standardized values\.

In [28]:
# ---------------------------------------------------------------------
# Export compact support and scaler summaries
# ---------------------------------------------------------------------

support_summary_df = (
    clustering_metadata_df
    .groupby(
        ["pre_post_cp", "support_issue_pattern"],
        dropna=False,
        as_index=False,
    )
    .agg(
        zone_period_rows=("taxi_zone_id", "count"),
        mean_support_issue_count=("support_issue_count", "mean"),
    )
    .sort_values(
        ["pre_post_cp", "zone_period_rows", "support_issue_pattern"],
        ascending=[True, False, True],
    )
    .reset_index(drop=True)
)

if SAVE_CLUSTERING_INPUTS:
    support_summary_df.to_csv(
        SUPPORT_SUMMARY_PATH,
        index=False,
    )
    scaler_summary_df.to_csv(
        SCALER_SUMMARY_PATH,
        index=False,
    )

print("Support and scaler summaries exported.")

exported_summary_tables_df = pd.DataFrame(
    {
        "artifact": [
            "support_summary",
            "scaler_summary",
        ],
        "path": [
            str(SUPPORT_SUMMARY_PATH),
            str(SCALER_SUMMARY_PATH),
        ],
        "exists_after_run": [
            SUPPORT_SUMMARY_PATH.exists(),
            SCALER_SUMMARY_PATH.exists(),
        ],
    }
)

display(exported_summary_tables_df)

Support and scaler summaries exported.


,artifact,path,exists_after_run
0,support_summary,pipeline_data/3.1.1.final_tables/mobility_prof...,True
1,scaler_summary,pipeline_data/3.1.1.final_tables/mobility_prof...,True


### Write clustering feature column list

Export a skinny machine\-readable feature list that identifies which columns belong to the primary clustering matrix, which feature family they came from, and which source metric they represent\. This is for downstream execution, not narrative documentation\.

In [29]:
# ---------------------------------------------------------------------
# Export the machine-readable clustering feature column list
# ---------------------------------------------------------------------

feature_column_records = []

for metric in PRIMARY_CLUSTERING_METRICS:
    feature_column_records.append(
        {
            "feature_column": make_level_feature_name(metric),
            "feature_family": "period_level",
            "source_metric": metric,
            "temporal_bucket": pd.NA,
        }
    )

for metric in TEMPORAL_SHAPE_FEATURE_METRICS:
    for temporal_bucket in TEMPORAL_BUCKET_ORDER:
        feature_column_records.append(
            {
                "feature_column": make_shape_feature_name(metric, temporal_bucket),
                "feature_family": "temporal_shape",
                "source_metric": metric,
                "temporal_bucket": temporal_bucket,
            }
        )

clustering_feature_columns_df = pd.DataFrame(feature_column_records)

if SAVE_CLUSTERING_INPUTS:
    clustering_feature_columns_df.to_csv(
        FEATURE_COLUMN_LIST_PATH,
        index=False,
    )

print("Feature-column list exported.")

display(clustering_feature_columns_df.head(120))

Feature-column list exported.


,feature_column,feature_family,source_metric,temporal_bucket
0,taxi_trip_count_period_mean,period_level,taxi_trip_count,<NA>
1,taxi_avg_trip_speed_period_mean,period_level,taxi_avg_trip_speed,<NA>
2,taxi_avg_trip_duration_period_mean,period_level,taxi_avg_trip_duration,<NA>
3,fhvhv_trip_count_period_mean,period_level,fhvhv_trip_count,<NA>
4,fhvhv_avg_trip_speed_period_mean,period_level,fhvhv_avg_trip_speed,<NA>
5,fhvhv_avg_trip_duration_period_mean,period_level,fhvhv_avg_trip_duration,<NA>
6,subway_ridership_period_mean,period_level,subway_ridership,<NA>
7,subway_transfers_period_mean,period_level,subway_transfers,<NA>
8,bus_trip_count_period_mean,period_level,bus_trip_count,<NA>
9,avg_bus_speed_period_mean,period_level,avg_bus_speed,<NA>


### Validate final handoff

Confirm all expected output files exist, row counts align, primary keys are unique, and the final matrix is ready for 3\.2\.1\.

In [30]:
# ---------------------------------------------------------------------
# Validate exported clustering handoff artifacts
# ---------------------------------------------------------------------

raw_export_df = pd.read_parquet(RAW_CLUSTERING_FEATURE_TABLE_PATH)
scaled_export_df = pd.read_parquet(SCALED_CLUSTERING_MATRIX_PATH)
metadata_export_df = pd.read_parquet(CLUSTERING_METADATA_PATH)

final_export_validation_df = pd.DataFrame(
    {
        "check": [
            "raw_feature_export_exists",
            "scaled_matrix_export_exists",
            "metadata_export_exists",
            "raw_scaled_metadata_row_counts_match",
            "raw_export_unique_zone_period_key",
            "scaled_export_unique_zone_period_key",
            "metadata_export_unique_zone_period_key",
            "scaled_feature_columns_match_contract",
            "scaled_export_has_no_null_features",
        ],
        "status": [
            RAW_CLUSTERING_FEATURE_TABLE_PATH.exists(),
            SCALED_CLUSTERING_MATRIX_PATH.exists(),
            CLUSTERING_METADATA_PATH.exists(),
            len(raw_export_df) == len(scaled_export_df) == len(metadata_export_df),
            len(raw_export_df) == len(raw_export_df[CLUSTERING_GRAIN_COLUMNS].drop_duplicates()),
            len(scaled_export_df) == len(scaled_export_df[CLUSTERING_GRAIN_COLUMNS].drop_duplicates()),
            len(metadata_export_df) == len(metadata_export_df[CLUSTERING_GRAIN_COLUMNS].drop_duplicates()),
            scaling_feature_columns == [
                column
                for column in scaled_export_df.columns
                if column not in CLUSTERING_GRAIN_COLUMNS
            ],
            not scaled_export_df[scaling_feature_columns].isna().any().any(),
        ],
    }
)

display(final_export_validation_df)

if not final_export_validation_df["status"].all():
    failed_checks = final_export_validation_df.loc[
        ~final_export_validation_df["status"],
        "check",
    ].tolist()
    raise AssertionError(
        f"3.1.1 final export validation failed: {failed_checks}"
    )

,check,status
0,raw_feature_export_exists,True
1,scaled_matrix_export_exists,True
2,metadata_export_exists,True
3,raw_scaled_metadata_row_counts_match,True
4,raw_export_unique_zone_period_key,True
5,scaled_export_unique_zone_period_key,True
6,metadata_export_unique_zone_period_key,True
7,scaled_feature_columns_match_contract,True
8,scaled_export_has_no_null_features,True


## Close

This notebook turns the 1.3.1 mobility panel into a clean retrospective clustering input at the Taxi Zone × pre_post_cp grain. We kept the full city landscape, preserved the 10 retained mobility metrics, separated overall level from temporal shape, and carried support context alongside the numeric matrix instead of hiding sparse-service cases. By the end, the important work is done: Chapter 3 now has a raw clustering feature table, a scaled matrix, and aligned metadata that are clean enough to move straight into candidate cluster generation and validation.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=5c43663c-345d-4c21-9332-dd4f928af3ba' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>